<a href="https://colab.research.google.com/github/osung/apollo.M1.GNN/blob/main/apollo_M1_GNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!git clone https://github.com/osung/apollo.M1.GNN.git

Cloning into 'apollo.M1.GNN'...
remote: Enumerating objects: 305, done.
remote: Counting objects: 100% (305/305), done.
remote: Compressing objects: 100% (185/185), done.
remote: Total 305 (delta 169), reused 255 (delta 119), pack-reused 0 (from 0)
Receiving objects: 100% (305/305), 141.73 KiB | 2.67 MiB/s, done.
Resolving deltas: 100% (169/169), done.


In [3]:
%cd apollo.M1.GNN

/content/apollo.M1.GNN


In [4]:
!pip install torch-geometric faiss-cpu pyyaml --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 86.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 116.7 MB/s eta 0:00:00


In [5]:

import torch, torch_geometric
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())
print('pyg:', torch_geometric.__version__)

torch: 2.10.0+cu128 cuda: True
pyg: 2.7.0


In [6]:

import os
os.makedirs('data/processed', exist_ok=True)

In [7]:
# graph + held_out + node_maps는 Drive에서 심볼릭 링크
!ln -sf /content/drive/MyDrive/apollo.M1.GNN/held_out.pt       data/processed/held_out.pt
!ln -sf /content/drive/MyDrive/apollo.M1.GNN/node_maps.pkl     data/processed/node_maps.pkl
!ln -sf /content/drive/MyDrive/apollo.M1.GNN/sim_edges_top5.npz data/processed/sim_edges_top5.npz
!ln -sf /content/drive/MyDrive/apollo.M1.GNN/sim_edges_top10.npz data/processed/sim_edges_top10.npz

In [8]:
!ln -sf /content/drive/MyDrive/apollo.M1.GNN/sim_edges_top20.npz data/processed/sim_edges_top20.npz
!ln -sf /content/drive/MyDrive/apollo.M1.GNN/sim_edges_top50.npz data/processed/sim_edges_top50.npz
!ln -sf /content/drive/MyDrive/apollo.M1.GNN/sim_edges_top100.npz data/processed/sim_edges_top100.npz

In [9]:
!ln -sf /content/drive/MyDrive/apollo.M1.GNN/hard_negatives_p2c_clean.npz data/processed/hard_negatives_p2c_clean.npz
!ln -sf /content/drive/MyDrive/apollo.M1.GNN/hard_negatives_c2p_clean.npz data/processed/hard_negatives_c2p_clean.npz

In [ ]:
!python scripts/train_gnn.py --graph-path /content/drive/MyDrive/apollo.M1.GNN/graph_fp16.pt --held-out-path data/processed/held_out.pt --with-similarity --sim-path data/processed/sim_edges_top10.npz --layer-type sage --hidden-dim 64 --num-layers 2 --epochs 20 --device cuda --checkpoint-every 5 --save-model-ckpt --eval-at-checkpoint --direction both --save-metrics data/processed/checkpoints/gnn_metrics_phase2.json --checkpoint-dir data/processed/checkpoints\



[train_gnn] loading graph /content/drive/MyDrive/apollo.M1.GNN/graph_fp16.pt
[train_gnn] casting project.x torch.float16 -> torch.float32
[train_gnn] casting company.x torch.float16 -> torch.float32
[train_gnn] loading similarity edges data/processed/sim_edges_top10.npz
[train_gnn] added similarity edges: |E|=12,138,565  weight=0.25  reverse=True
[train_gnn] layer_type=sage  params=247,168
[train_gnn] epochs=20 batch_size=4096 num_neg=5 lr=0.001 wd=1e-05 device=cuda
[train_gnn] checkpointing every 5 epochs -> data/processed/checkpoints
[train] epoch   1/20  loss=0.2321  time=7.1s
[train] epoch   2/20  loss=0.1058  time=6.3s
[train] epoch   3/20  loss=0.0994  time=6.3s
[train] epoch   4/20  loss=0.0976  time=6.3s
[train] epoch   5/20  loss=0.0967  time=6.3s
[ckpt] epoch 5: saved project_z_sage_h64_l2_epoch005.npy, company_z_sage_h64_l2_epoch005.npy
[ckpt] epoch 5: saved model_sage_h64_l2_epoch005.pt

=== GNN [sage] epoch 5 [p2c] K=100 ===
  royalty       recall@10=0.0010  ndcg@10=0.0003

In [ ]:
!python scripts/train_gnn.py --graph-path /content/drive/MyDrive/apollo.M1.GNN/graph.pt --held-out-path data/processed/held_out.pt --with-similarity --sim-path data/processed/sim_edges_top10.npz --layer-type gcn --hidden-dim 64 --num-layers 2 --epochs 100 --device cuda --checkpoint-every 5 --save-model-ckpt --eval-at-checkpoint --direction both --save-metrics data/processed/checkpoints/gnn_metrics_phase2.json --checkpoint-dir /content/drive/MyDrive/apollo.M1.GNN/outputs


[train_gnn] loading graph /content/drive/MyDrive/apollo.M1.GNN/graph.pt
[train_gnn] loading similarity edges data/processed/sim_edges_top10.npz
[train_gnn] added similarity edges: |E|=12,138,565  weight=0.25  reverse=True
[train_gnn] layer_type=gcn  params=247,168  normalize_output=True
[train_gnn] epochs=100 batch_size=4096 num_neg=5 (p2c weight=1.0, n_hard=0, n_random=5; c2p enabled=False, weight=0.0, n_hard=0, n_random=5) lr=0.001 wd=1e-05 device=cuda
[train_gnn] experiment tag: gcn_h64_l2_sim10
[train_gnn] checkpointing every 5 epochs -> /content/drive/MyDrive/apollo.M1.GNN/outputs
[train] epoch   1/100  loss=0.2321  time=6.7s
[train] epoch   2/100  loss=0.1058  time=6.3s
[train] epoch   3/100  loss=0.0994  time=6.3s
[train] epoch   4/100  loss=0.0976  time=6.3s
[train] epoch   5/100  loss=0.0967  time=6.3s
[ckpt] epoch 5: saved project_z_gcn_h64_l2_sim10_epoch005.npy, company_z_gcn_h64_l2_sim10_epoch005.npy
[ckpt] epoch 5: saved model_gcn_h64_l2_sim10_epoch005.pt

=== GNN [gcn] ep

In [ ]:
!python scripts/train_gnn.py --graph-path /content/drive/MyDrive/apollo.M1.GNN/graph_fp16.pt --held-out-path data/processed/held_out.pt --with-similarity --sim-path data/processed/sim_edges_top10.npz --layer-type hgt --hidden-dim 64 --num-layers 2 --epochs 20 --device cuda --checkpoint-every 5 --save-model-ckpt --eval-at-checkpoint --direction both --save-metrics data/processed/checkpoints/gnn_metrics_phase2.json --checkpoint-dir data/processed/checkpoints\


[train_gnn] loading graph /content/drive/MyDrive/apollo.M1.GNN/graph_fp16.pt
[train_gnn] casting project.x torch.float16 -> torch.float32
[train_gnn] casting company.x torch.float16 -> torch.float32
[train_gnn] loading similarity edges data/processed/sim_edges_top10.npz
[train_gnn] added similarity edges: |E|=12,138,565  weight=0.25  reverse=True
[train_gnn] layer_type=hgt  params=214,468
[train_gnn] epochs=20 batch_size=4096 num_neg=5 lr=0.001 wd=1e-05 device=cuda
[train_gnn] checkpointing every 5 epochs -> data/processed/checkpoints\
[train] epoch   1/20  loss=0.4178  time=35.4s
[train] epoch   2/20  loss=0.2256  time=33.8s
[train] epoch   3/20  loss=0.1034  time=34.1s
[train] epoch   4/20  loss=0.0961  time=34.2s
[train] epoch   5/20  loss=0.0951  time=34.0s
[ckpt] epoch 5: saved project_z_hgt_h64_l2_epoch005.npy, company_z_hgt_h64_l2_epoch005.npy
[ckpt] epoch 5: saved model_hgt_h64_l2_epoch005.pt

=== GNN [hgt] epoch 5 [p2c] K=100 ===
  royalty       recall@10=0.0000  ndcg@10=0.000

In [ ]:
!git pull origin main

From https://github.com/osung/apollo.M1.GNN
 * branch            main       -> FETCH_HEAD
Already up to date.


In [ ]:
!python scripts/train_gnn.py --graph-path /content/drive/MyDrive/apollo.M1.GNN/graph_fp16.pt --held-out-path data/processed/held_out.pt --with-similarity --sim-path data/processed/sim_edges_top10.npz --hard-neg-path data/processed/hard_negatives_p2c.npz --hard-ratio 0.5 --num-neg 20 --layer-type hgt --hidden-dim 64 --num-layers 2 --epochs 20 --device cuda --checkpoint-every 5 --save-model-ckpt --eval-at-checkpoint --direction both --save-metrics data/processed/checkpoints/gnn_metrics_phase2.json --checkpoint-dir data/processed/checkpoints\

[train_gnn] loading graph /content/drive/MyDrive/apollo.M1.GNN/graph_fp16.pt
[train_gnn] casting project.x torch.float16 -> torch.float32
[train_gnn] casting company.x torch.float16 -> torch.float32
[train_gnn] loading similarity edges data/processed/sim_edges_top10.npz
[train_gnn] added similarity edges: |E|=12,138,565  weight=0.25  reverse=True
[train_gnn] layer_type=hgt  params=214,468
[train_gnn] loading hard negatives data/processed/hard_negatives_p2c.npz
[train_gnn]   430,374 projects have hard-neg pool  (min=20, median=20, max=20)  hard_ratio=0.5
[train_gnn] epochs=20 batch_size=4096 num_neg=20 (n_hard=10, n_random=10) lr=0.001 wd=1e-05 device=cuda
[train_gnn] checkpointing every 5 epochs -> data/processed/checkpoints\
[train] epoch   1/20  loss=0.4302  time=35.7s
[train] epoch   2/20  loss=0.2164  time=34.9s
[train] epoch   3/20  loss=0.1276  time=35.0s
[train] epoch   4/20  loss=0.1240  time=34.9s
[train] epoch   5/20  loss=0.1228  time=35.0s
[ckpt] epoch 5: saved project_z_hg

In [ ]:
!python scripts/train_gnn.py --graph-path /content/drive/MyDrive/apollo.M1.GNN/graph_fp16.pt --held-out-path data/processed/held_out.pt --with-similarity --sim-path data/processed/sim_edges_top10.npz --hard-neg-path data/processed/hard_negatives_p2c.npz --hard-ratio 0.5 --num-neg 20 --layer-type hgt --hidden-dim 64 --num-layers 2 --epochs 20 --device cuda --checkpoint-every 5 --save-model-ckpt --eval-at-checkpoint --direction both --save-metrics data/processed/checkpoints/gnn_metrics_phase2.json --checkpoint-dir data/processed/checkpoints\

[train_gnn] loading graph /content/drive/MyDrive/apollo.M1.GNN/graph_fp16.pt
[train_gnn] casting project.x torch.float16 -> torch.float32
[train_gnn] casting company.x torch.float16 -> torch.float32
[train_gnn] loading similarity edges data/processed/sim_edges_top10.npz
[train_gnn] added similarity edges: |E|=12,138,565  weight=0.25  reverse=True
[train_gnn] layer_type=hgt  params=214,468  normalize_output=True
[train_gnn] loading hard negatives data/processed/hard_negatives_p2c.npz
[train_gnn]   430,374 projects have hard-neg pool  (min=20, median=20, max=20)  hard_ratio=0.5
[train_gnn] epochs=20 batch_size=4096 num_neg=20 (n_hard=10, n_random=10) lr=0.001 wd=1e-05 device=cuda
[train_gnn] checkpointing every 5 epochs -> data/processed/checkpoints\
[train] epoch   1/20  loss=0.4302  time=35.8s
[train] epoch   2/20  loss=0.2164  time=34.9s
[train] epoch   3/20  loss=0.1276  time=34.8s
[train] epoch   4/20  loss=0.1240  time=34.9s
[train] epoch   5/20  loss=0.1228  time=34.9s
[ckpt] epoc

In [ ]:
!python scripts/train_gnn.py --graph-path /content/drive/MyDrive/apollo.M1.GNN/graph_fp16.pt --held-out-path data/processed/held_out.pt --with-similarity --sim-path data/processed/sim_edges_top10.npz --hard-neg-path data/processed/hard_negatives_p2c.npz --hard-ratio 0.5 --num-neg 20 --layer-type hgt --hidden-dim 64 --num-layers 2 --epochs 100 --device cuda --checkpoint-every 10 --save-model-ckpt --eval-at-checkpoint --direction both --save-metrics data/processed/checkpoints/gnn_metrics_phase2.json --checkpoint-dir data/processed/checkpoints\

[train_gnn] loading graph /content/drive/MyDrive/apollo.M1.GNN/graph_fp16.pt
[train_gnn] casting project.x torch.float16 -> torch.float32
[train_gnn] casting company.x torch.float16 -> torch.float32
[train_gnn] loading similarity edges data/processed/sim_edges_top10.npz
[train_gnn] added similarity edges: |E|=12,138,565  weight=0.25  reverse=True
[train_gnn] layer_type=hgt  params=214,468  normalize_output=True
[train_gnn] loading hard negatives data/processed/hard_negatives_p2c.npz
[train_gnn]   430,374 projects have hard-neg pool  (min=20, median=20, max=20)  hard_ratio=0.5
[train_gnn] epochs=100 batch_size=4096 num_neg=20 (n_hard=10, n_random=10) lr=0.001 wd=1e-05 device=cuda
[train_gnn] checkpointing every 10 epochs -> data/processed/checkpoints\
[train] epoch   1/100  loss=0.4302  time=35.5s
[train] epoch   2/100  loss=0.2164  time=34.7s
[train] epoch   3/100  loss=0.1276  time=35.1s
[train] epoch   4/100  loss=0.1240  time=34.9s
[train] epoch   5/100  loss=0.1228  time=35.0s
[tra

In [ ]:
!python scripts/train_gnn.py --graph-path /content/drive/MyDrive/apollo.M1.GNN/graph_fp16.pt --held-out-path data/processed/held_out.pt --with-similarity --sim-path data/processed/sim_edges_top10.npz --hard-neg-path data/processed/hard_negatives_p2c.npz --hard-ratio 0.5 --num-neg 20  --no-normalize --layer-type hgt --hidden-dim 64 --num-layers 2 --epochs 100 --device cuda --checkpoint-every 5 --save-model-ckpt --eval-at-checkpoint --direction both --save-metrics data/processed/checkpoints/gnn_metrics_phase2.json --checkpoint-dir data/processed/checkpoints\

[train_gnn] loading graph /content/drive/MyDrive/apollo.M1.GNN/graph_fp16.pt
[train_gnn] casting project.x torch.float16 -> torch.float32
[train_gnn] casting company.x torch.float16 -> torch.float32
[train_gnn] loading similarity edges data/processed/sim_edges_top10.npz
[train_gnn] added similarity edges: |E|=12,138,565  weight=0.25  reverse=True
[train_gnn] layer_type=hgt  params=214,468  normalize_output=False
[train_gnn] loading hard negatives data/processed/hard_negatives_p2c.npz
[train_gnn]   430,374 projects have hard-neg pool  (min=20, median=20, max=20)  hard_ratio=0.5
[train_gnn] epochs=100 batch_size=4096 num_neg=20 (n_hard=10, n_random=10) lr=0.001 wd=1e-05 device=cuda
[train_gnn] checkpointing every 5 epochs -> data/processed/checkpoints\
[train] epoch   1/100  loss=0.4320  time=35.2s
[train] epoch   2/100  loss=0.1263  time=34.7s
[train] epoch   3/100  loss=0.0313  time=34.8s
[train] epoch   4/100  loss=0.0276  time=34.7s
[train] epoch   5/100  loss=0.0267  time=34.7s
[ckp

[train_gnn] loading graph /content/drive/MyDrive/apollo.M1.GNN/graph_fp16.pt
[train_gnn] casting project.x torch.float16 -> torch.float32
[train_gnn] casting company.x torch.float16 -> torch.float32
[train_gnn] loading similarity edges data/processed/sim_edges_top10.npz
[train_gnn] added similarity edges: |E|=12,138,565  weight=0.25  reverse=True
[train_gnn] layer_type=hgt  params=214,468  normalize_output=False
[train_gnn] loading hard negatives data/processed/hard_negatives_p2c.npz
[train_gnn]   430,374 projects have hard-neg pool  (min=20, median=20, max=20)  hard_ratio=0.5
[train_gnn] epochs=100 batch_size=4096 num_neg=20 (n_hard=10, n_random=10) lr=0.001 wd=1e-05 device=cuda
[train_gnn] checkpointing every 5 epochs -> data/processed/checkpoints\
[train] epoch   1/100  loss=0.4320  time=35.2s
[train] epoch   2/100  loss=0.1263  time=34.7s
[train] epoch   3/100  loss=0.0313  time=34.8s
[train] epoch   4/100  loss=0.0276  time=34.7s
[train] epoch   5/100  loss=0.0267  time=34.7s
[ckpt] epoch 5: saved project_z_hgt_h64_l2_epoch005.npy, company_z_hgt_h64_l2_epoch005.npy
[ckpt] epoch 5: saved model_hgt_h64_l2_epoch005.pt

=== GNN [hgt] epoch 5 [p2c] K=100 ===
  royalty       recall@10=0.0073  ndcg@10=0.0036  recall@100=0.0392  ndcg@100=0.0100
  commercial    recall@10=0.0020  ndcg@10=0.0009  recall@100=0.0199  ndcg@100=0.0042
  performance   recall@10=0.0195  ndcg@10=0.0132  recall@100=0.0517  ndcg@100=0.0193


In [ ]:
!git pull origin main

remote: Enumerating objects: 26, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 17 (delta 14), reused 17 (delta 14), pack-reused 0 (from 0)
Unpacking objects: 100% (17/17), 4.95 KiB | 1014.00 KiB/s, done.
From https://github.com/osung/apollo.M1.GNN
 * branch            main       -> FETCH_HEAD
   d984f32..a69fb76  main       -> origin/main
Updating d984f32..a69fb76
Fast-forward
 scripts/train_gnn.py    |  57 +++++++++++++++++++----
 src/training/sampler.py | 117 ++++++++++++++++++++++++++++++------------------
 src/training/trainer.py |  12 ++++-
 tests/test_sampler.py   |  59 ++++++++++++++++++++++++
 4 files changed, 192 insertions(+), 53 deletions(-)


In [ ]:
!git pull origin main && python scripts/train_gnn.py --graph-path /content/drive/MyDrive/apollo.M1.GNN/graph.pt --held-out-path data/processed/held_out.pt --with-similarity --sim-path data/processed/sim_edges_top10.npz --hard-neg-path data/processed/hard_negatives_p2c.npz --hard-neg-path-c2p data/processed/hard_negatives_c2p.npz --p2c-weight 1.0 --c2p-weight 1.0 --hard-ratio 0.7 --num-neg 20 --no-normalize --layer-type hgt --hidden-dim 64 --num-layers 2 --num-heads 4 --epochs 100 --device cuda --checkpoint-every 5 --save-model-ckpt --eval-at-checkpoint --direction both --save-metrics data/processed/checkpoints/gnn_metrics_hr07.json --checkpoint-dir data/processed/checkpoints

From https://github.com/osung/apollo.M1.GNN
 * branch            main       -> FETCH_HEAD
Already up to date.
[train_gnn] loading graph /content/drive/MyDrive/apollo.M1.GNN/graph.pt
[train_gnn] loading similarity edges data/processed/sim_edges_top10.npz
[train_gnn] added similarity edges: |E|=12,138,565  weight=0.25  reverse=True
[train_gnn] layer_type=hgt  params=214,468  normalize_output=False
[train_gnn] loading p2c hard negatives data/processed/hard_negatives_p2c.npz
[train_gnn]   p2c: 430,374 projects  (min=20, median=20, max=20)
[train_gnn] loading c2p hard negatives data/processed/hard_negatives_c2p.npz
[train_gnn]   c2p: 817,117 companies  (min=20, median=20, max=20)
[train_gnn] epochs=100 batch_size=4096 num_neg=20 (p2c weight=1.0, n_hard=14, n_random=6; c2p enabled=True, weight=1.0, n_hard=14, n_random=6) lr=0.001 wd=1e-05 device=cuda
[train_gnn] checkpointing every 5 epochs -> data/processed/checkpoints
[train] epoch   1/100  loss=0.8882  time=36.0s
[train] epoch   2/100  lo

In [ ]:
!git pull origin main && python scripts/train_gnn.py --graph-path /content/drive/MyDrive/apollo.M1.GNN/graph.pt --held-out-path data/processed/held_out.pt --with-similarity --sim-path data/processed/sim_edges_top10.npz --hard-neg-path data/processed/hard_negatives_p2c.npz --hard-neg-path-c2p data/processed/hard_negatives_c2p.npz --p2c-weight 1.0 --c2p-weight 1.0 --hard-ratio 0.7 --num-neg 20 --no-normalize --layer-type hgt --hidden-dim 128 --num-layers 2 --num-heads 4 --epochs 100 --device cuda --checkpoint-every 5 --save-model-ckpt --eval-at-checkpoint --direction both --save-metrics data/processed/checkpoints/gnn_metrics_hr07.json --checkpoint-dir data/processed/checkpoints.hd128

From https://github.com/osung/apollo.M1.GNN
 * branch            main       -> FETCH_HEAD
Already up to date.
[train_gnn] loading graph /content/drive/MyDrive/apollo.M1.GNN/graph.pt
[train_gnn] loading similarity edges data/processed/sim_edges_top10.npz
[train_gnn] added similarity edges: |E|=12,138,565  weight=0.25  reverse=True
[train_gnn] layer_type=hgt  params=625,220  normalize_output=False
[train_gnn] loading p2c hard negatives data/processed/hard_negatives_p2c.npz
[train_gnn]   p2c: 430,374 projects  (min=20, median=20, max=20)
[train_gnn] loading c2p hard negatives data/processed/hard_negatives_c2p.npz
[train_gnn]   c2p: 817,117 companies  (min=20, median=20, max=20)
[train_gnn] epochs=100 batch_size=4096 num_neg=20 (p2c weight=1.0, n_hard=14, n_random=6; c2p enabled=True, weight=1.0, n_hard=14, n_random=6) lr=0.001 wd=1e-05 device=cuda
[train_gnn] checkpointing every 5 epochs -> data/processed/checkpoints.hd128
Traceback (most recent call last):
  File "/content/apollo.M1.GNN/

In [ ]:
!git pull origin main && python scripts/train_gnn.py --graph-path /content/drive/MyDrive/apollo.M1.GNN/graph.pt --held-out-path data/processed/held_out.pt --with-similarity --sim-path data/processed/sim_edges_top10.npz --hard-neg-path data/processed/hard_negatives_p2c.npz --hard-neg-path-c2p data/processed/hard_negatives_c2p.npz --p2c-weight 1.0 --c2p-weight 1.0 --hard-ratio 0.8 --num-neg 20 --no-normalize --layer-type hgt --hidden-dim 64 --num-layers 2 --num-heads 4 --epochs 100 --device cuda --checkpoint-every 5 --save-model-ckpt --eval-at-checkpoint --direction both --save-metrics data/processed/checkpoints/gnn_metrics_hr07.json --checkpoint-dir data/processed/checkpoints.hd128

From https://github.com/osung/apollo.M1.GNN
 * branch            main       -> FETCH_HEAD
Already up to date.
[train_gnn] loading graph /content/drive/MyDrive/apollo.M1.GNN/graph.pt
[train_gnn] loading similarity edges data/processed/sim_edges_top10.npz
[train_gnn] added similarity edges: |E|=12,138,565  weight=0.25  reverse=True
[train_gnn] layer_type=hgt  params=214,468  normalize_output=False
[train_gnn] loading p2c hard negatives data/processed/hard_negatives_p2c.npz
[train_gnn]   p2c: 430,374 projects  (min=20, median=20, max=20)
[train_gnn] loading c2p hard negatives data/processed/hard_negatives_c2p.npz
[train_gnn]   c2p: 817,117 companies  (min=20, median=20, max=20)
[train_gnn] epochs=100 batch_size=4096 num_neg=20 (p2c weight=1.0, n_hard=16, n_random=4; c2p enabled=True, weight=1.0, n_hard=16, n_random=4) lr=0.001 wd=1e-05 device=cuda
[train_gnn] checkpointing every 5 epochs -> data/processed/checkpoints.hd128
[train] epoch   1/100  loss=0.8879  time=36.8s
[train] epoch   2/1

=== GNN [hgt] epoch 90 [p2c] K=100 ===
  royalty       recall@10=0.1260  ndcg@10=0.0633  recall@100=0.3583  ndcg@100=0.1110
  commercial    recall@10=0.1107  ndcg@10=0.0571  recall@100=0.3323  ndcg@100=0.1021
  performance   recall@10=0.0996  ndcg@10=0.0501  recall@100=0.2832  ndcg@100=0.0872

=== GNN [hgt] epoch 90 [c2p] K=100 ===
  royalty       recall@10=0.0195  ndcg@10=0.0075  recall@100=0.1756  ndcg@100=0.0367
  commercial    recall@10=0.0150  ndcg@10=0.0074  recall@100=0.1271  ndcg@100=0.0292
  performance   recall@10=0.0208  ndcg@10=0.0098  recall@100=0.1393  ndcg@100=0.0330
[ckpt] epoch 90: saved metrics_hgt_h64_l2_epoch090.json
[train] epoch  91/100  loss=0.0034  time=35.9s
[train] epoch  92/100  loss=0.0030  time=36.0s
[train] epoch  93/100  loss=0.0029  time=35.8s
[train] epoch  94/100  loss=0.0039  time=35.8s
[train] epoch  95/100  loss=0.0031  time=35.8s
[ckpt] epoch 95: saved project_z_hgt_h64_l2_epoch095.npy, company_z_hgt_h64_l2_epoch095.npy
[ckpt] epoch 95: saved model_hgt_h64_l2_epoch095.pt

=== GNN [hgt] epoch 95 [p2c] K=100 ===
  royalty       recall@10=0.1663  ndcg@10=0.0843  recall@100=0.3849  ndcg@100=0.1287
  commercial    recall@10=0.1236  ndcg@10=0.0633  recall@100=0.3575  ndcg@100=0.1113
  performance   recall@10=0.1196  ndcg@10=0.0615  recall@100=0.2987  ndcg@100=0.0980

=== GNN [hgt] epoch 95 [c2p] K=100 ===
  royalty       recall@10=0.0226  ndcg@10=0.0097  recall@100=0.2150  ndcg@100=0.0469
  commercial    recall@10=0.0206  ndcg@10=0.0092  recall@100=0.1505  ndcg@100=0.0343
  performance   recall@10=0.0252  ndcg@10=0.0124  recall@100=0.1629  ndcg@100=0.0398
[ckpt] epoch 95: saved metrics_hgt_h64_l2_epoch095.json
[train] epoch  96/100  loss=0.0029  time=35.9s
[train] epoch  97/100  loss=0.0030  time=35.8s
[train] epoch  98/100  loss=0.0029  time=35.8s
[train] epoch  99/100  loss=0.0033  time=35.8s
[train] epoch 100/100  loss=0.0029  time=35.9s
[train_gnn] saved project z -> data/processed/project_z.npy
[train_gnn] saved company z -> data/processed/company_z.npy

=== GNN [hgt] final [p2c] K=100 ===
  royalty       recall@10=0.1297  ndcg@10=0.0623  recall@100=0.3808  ndcg@100=0.1134
  commercial    recall@10=0.1166  ndcg@10=0.0597  recall@100=0.3430  ndcg@100=0.1052
  performance   recall@10=0.0977  ndcg@10=0.0509  recall@100=0.2970  ndcg@100=0.0911

=== GNN [hgt] final [c2p] K=100 ===
  royalty       recall@10=0.0447  ndcg@10=0.0213  recall@100=0.2385  ndcg@100=0.0590
  commercial    recall@10=0.0235  ndcg@10=0.0099  recall@100=0.1540  ndcg@100=0.0357
  performance   recall@10=0.0338  ndcg@10=0.0173  recall@100=0.1855  ndcg@100=0.0473
[train_gnn] saved metrics -> data/processed/checkpoints/gnn_metrics_hr07.json

In [ ]:
!git pull origin main && python scripts/train_gnn.py --graph-path /content/drive/MyDrive/apollo.M1.GNN/graph.pt --held-out-path data/processed/held_out.pt --with-similarity --sim-path data/processed/sim_edges_top10.npz --hard-neg-path data/processed/hard_negatives_p2c.npz --hard-neg-path-c2p data/processed/hard_negatives_c2p.npz --p2c-weight 1.0 --c2p-weight 1.0 --hard-ratio 0.8 --num-neg 20 --no-normalize --layer-type hgt --hidden-dim 64 --num-layers 2 --num-heads 4 --epochs 120 --device cuda --checkpoint-every 5 --save-model-ckpt --eval-at-checkpoint --direction both --save-metrics data/processed/checkpoints/gnn_metrics_hr08.json --checkpoint-dir data/processed/checkpoints.hd128 --resume data/processed/checkpoints.hd128/model_hgt_h64_l2_epoch095.pt

From https://github.com/osung/apollo.M1.GNN
 * branch            main       -> FETCH_HEAD
Already up to date.
[train_gnn] loading graph /content/drive/MyDrive/apollo.M1.GNN/graph.pt
[train_gnn] loading similarity edges data/processed/sim_edges_top10.npz
[train_gnn] added similarity edges: |E|=12,138,565  weight=0.25  reverse=True
[train_gnn] layer_type=hgt  params=214,468  normalize_output=False
[train_gnn] loading p2c hard negatives data/processed/hard_negatives_p2c.npz
[train_gnn]   p2c: 430,374 projects  (min=20, median=20, max=20)
[train_gnn] loading c2p hard negatives data/processed/hard_negatives_c2p.npz
[train_gnn]   c2p: 817,117 companies  (min=20, median=20, max=20)
[train_gnn] epochs=120 batch_size=4096 num_neg=20 (p2c weight=1.0, n_hard=16, n_random=4; c2p enabled=True, weight=1.0, n_hard=16, n_random=4) lr=0.001 wd=1e-05 device=cuda
[train_gnn] resuming from data/processed/checkpoints.hd128/model_hgt_h64_l2_epoch095.pt
[train_gnn] resumed at epoch 95, continuing to 120
[tra

In [ ]:
!git pull origin main

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 764 bytes | 764.00 KiB/s, done.
From https://github.com/osung/apollo.M1.GNN
 * branch            main       -> FETCH_HEAD
   c31a1b6..58dc5db  main       -> origin/main
Updating c31a1b6..58dc5db
Fast-forward
 scripts/train_gnn.py | 7 +++++++
 1 file changed, 7 insertions(+)


In [ ]:
!git pull origin main && python scripts/train_gnn.py --graph-path /content/drive/MyDrive/apollo.M1.GNN/graph.pt --held-out-path data/processed/held_out.pt --with-similarity --sim-path data/processed/sim_edges_top10.npz --hard-neg-path data/processed/hard_negatives_p2c.npz --hard-neg-path-c2p data/processed/hard_negatives_c2p.npz --p2c-weight 1.0 --c2p-weight 1.0 --hard-ratio 0.8 --num-neg 20 --no-normalize --layer-type gat --hidden-dim 64 --num-layers 2 --num-heads 4 --epochs 120 --device cuda --checkpoint-every 5 --save-model-ckpt --eval-at-checkpoint --direction both --save-metrics data/processed/checkpoints/gnn_metrics_hr08.json --checkpoint-dir data/processed/checkpoints.hd128

From https://github.com/osung/apollo.M1.GNN
 * branch            main       -> FETCH_HEAD
Already up to date.
[train_gnn] loading graph /content/drive/MyDrive/apollo.M1.GNN/graph.pt
[train_gnn] loading similarity edges data/processed/sim_edges_top10.npz
[train_gnn] added similarity edges: |E|=12,138,565  weight=0.25  reverse=True
[train_gnn] layer_type=gat  params=648,576  normalize_output=False
[train_gnn] loading p2c hard negatives data/processed/hard_negatives_p2c.npz
[train_gnn]   p2c: 430,374 projects  (min=20, median=20, max=20)
[train_gnn] loading c2p hard negatives data/processed/hard_negatives_c2p.npz
[train_gnn]   c2p: 817,117 companies  (min=20, median=20, max=20)
[train_gnn] epochs=120 batch_size=4096 num_neg=20 (p2c weight=1.0, n_hard=16, n_random=4; c2p enabled=True, weight=1.0, n_hard=16, n_random=4) lr=0.001 wd=1e-05 device=cuda
[train_gnn] experiment tag: gat_h64_l2_nh4_hr80_c2p100_sim10_nonorm
[train_gnn] checkpointing every 5 epochs -> data/processed/checkpoints.hd12

[train_gnn] loading graph /content/drive/MyDrive/apollo.M1.GNN/graph.pt
[train_gnn] loading similarity edges data/processed/sim_edges_top10.npz
[train_gnn] added similarity edges: |E|=12,138,565  weight=0.25  reverse=True
[train_gnn] layer_type=gcn  params=247,168  normalize_output=False
[train_gnn] loading p2c hard negatives data/processed/hard_negatives_p2c.npz
[train_gnn]   p2c: 430,374 projects  (min=20, median=20, max=20)
[train_gnn] loading c2p hard negatives data/processed/hard_negatives_c2p.npz
[train_gnn]   c2p: 817,117 companies  (min=20, median=20, max=20)
[train_gnn] epochs=120 batch_size=4096 num_neg=20 (p2c weight=1.0, n_hard=16, n_random=4; c2p enabled=True, weight=1.0, n_hard=16, n_random=4) lr=0.001 wd=1e-05 device=cuda
[train_gnn] experiment tag: gcn_h64_l2_hr80_c2p100_sim10_nonorm
[train_gnn] checkpointing every 5 epochs -> data/processed/checkpoints.hd128
[train] epoch   1/120  loss=10188.0157  time=7.8s
[train] epoch   2/120  loss=1429.7536  time=7.4s
[train] epoch   3/120  loss=324.8638  time=7.4s
[train] epoch   4/120  loss=135.0579  time=7.4s
[train] epoch   5/120  loss=47.5764  time=7.4s
[ckpt] epoch 5: saved project_z_gcn_h64_l2_hr80_c2p100_sim10_nonorm_epoch005.npy, company_z_gcn_h64_l2_hr80_c2p100_sim10_nonorm_epoch005.npy
[ckpt] epoch 5: saved model_gcn_h64_l2_hr80_c2p100_sim10_nonorm_epoch005.pt

=== GNN [gcn] epoch 5 [p2c] K=100 ===
  royalty       recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0000  ndcg@100=0.0000
  commercial    recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0005  ndcg@100=0.0001
  performance   recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0002  ndcg@100=0.0000

=== GNN [gcn] epoch 5 [c2p] K=100 ===
  royalty       recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0000  ndcg@100=0.0000
  commercial    recall@10=0.0012  ndcg@10=0.0007  recall@100=0.0027  ndcg@100=0.0010
  performance   recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0001  ndcg@100=0.0000
[ckpt] epoch 5: saved metrics_gcn_h64_l2_hr80_c2p100_sim10_nonorm_epoch005.json
[train] epoch   6/120  loss=80.8262  time=7.4s
[train] epoch   7/120  loss=48.0222  time=7.4s
[train] epoch   8/120  loss=35.9349  time=7.4s
[train] epoch   9/120  loss=33.9686  time=7.4s
[train] epoch  10/120  loss=13.5073  time=7.4s
[ckpt] epoch 10: saved project_z_gcn_h64_l2_hr80_c2p100_sim10_nonorm_epoch010.npy, company_z_gcn_h64_l2_hr80_c2p100_sim10_nonorm_epoch010.npy
[ckpt] epoch 10: saved model_gcn_h64_l2_hr80_c2p100_sim10_nonorm_epoch010.pt

=== GNN [gcn] epoch 10 [p2c] K=100 ===
  royalty       recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0000  ndcg@100=0.0000
  commercial    recall@10=0.0003  ndcg@10=0.0001  recall@100=0.0003  ndcg@100=0.0001
  performance   recall@10=0.0002  ndcg@10=0.0001  recall@100=0.0005  ndcg@100=0.0001

=== GNN [gcn] epoch 10 [c2p] K=100 ===
  royalty       recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0011  ndcg@100=0.0002
  commercial    recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0005  ndcg@100=0.0001
  performance   recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0007  ndcg@100=0.0001
[ckpt] epoch 10: saved metrics_gcn_h64_l2_hr80_c2p100_sim10_nonorm_epoch010.json
[train] epoch  11/120  loss=10.5026  time=7.5s
[train] epoch  12/120  loss=8.3333  time=7.4s
[train] epoch  13/120  loss=8.1545  time=7.4s
[train] epoch  14/120  loss=7.3584  time=7.4s
[train] epoch  15/120  loss=5.4810  time=7.4s
[ckpt] epoch 15: saved project_z_gcn_h64_l2_hr80_c2p100_sim10_nonorm_epoch015.npy, company_z_gcn_h64_l2_hr80_c2p100_sim10_nonorm_epoch015.npy
[ckpt] epoch 15: saved model_gcn_h64_l2_hr80_c2p100_sim10_nonorm_epoch015.pt

=== GNN [gcn] epoch 15 [p2c] K=100 ===
  royalty       recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0000  ndcg@100=0.0000
  commercial    recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0003  ndcg@100=0.0000
  performance   recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0007  ndcg@100=0.0001

=== GNN [gcn] epoch 15 [c2p] K=100 ===
  royalty       recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0000  ndcg@100=0.0000
  commercial    recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0000  ndcg@100=0.0000
  performance   recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0007  ndcg@100=0.0002
[ckpt] epoch 15: saved metrics_gcn_h64_l2_hr80_c2p100_sim10_nonorm_epoch015.json
[train] epoch  16/120  loss=5.5185  time=7.5s
[train] epoch  17/120  loss=4.5981  time=7.4s
[train] epoch  18/120  loss=4.3752  time=7.4s
[train] epoch  19/120  loss=6.2269  time=7.4s
[train] epoch  20/120  loss=4.0656  time=7.4s
[ckpt] epoch 20: saved project_z_gcn_h64_l2_hr80_c2p100_sim10_nonorm_epoch020.npy, company_z_gcn_h64_l2_hr80_c2p100_sim10_nonorm_epoch020.npy
[ckpt] epoch 20: saved model_gcn_h64_l2_hr80_c2p100_sim10_nonorm_epoch020.pt

=== GNN [gcn] epoch 20 [p2c] K=100 ===
  royalty       recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0000  ndcg@100=0.0000
  commercial    recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0003  ndcg@100=0.0000
  performance   recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0005  ndcg@100=0.0001

=== GNN [gcn] epoch 20 [c2p] K=100 ===
  royalty       recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0011  ndcg@100=0.0002
  commercial    recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0002  ndcg@100=0.0000
  performance   recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0010  ndcg@100=0.0002
[ckpt] epoch 20: saved metrics_gcn_h64_l2_hr80_c2p100_sim10_nonorm_epoch020.json
[train] epoch  21/120  loss=4.0509  time=7.5s
[train] epoch  22/120  loss=3.1490  time=7.4s
[train] epoch  23/120  loss=3.1655  time=7.4s
[train] epoch  24/120  loss=2.7306  time=7.4s
[train] epoch  25/120  loss=2.1124  time=7.4s
[ckpt] epoch 25: saved project_z_gcn_h64_l2_hr80_c2p100_sim10_nonorm_epoch025.npy, company_z_gcn_h64_l2_hr80_c2p100_sim10_nonorm_epoch025.npy
[ckpt] epoch 25: saved model_gcn_h64_l2_hr80_c2p100_sim10_nonorm_epoch025.pt

=== GNN [gcn] epoch 25 [p2c] K=100 ===
  royalty       recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0000  ndcg@100=0.0000
  commercial    recall@10=0.0003  ndcg@10=0.0001  recall@100=0.0003  ndcg@100=0.0001
  performance   recall@10=0.0002  ndcg@10=0.0001  recall@100=0.0002  ndcg@100=0.0001

=== GNN [gcn] epoch 25 [c2p] K=100 ===
  royalty       recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0000  ndcg@100=0.0000
  commercial    recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0001  ndcg@100=0.0000
  performance   recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0010  ndcg@100=0.0002
[ckpt] epoch 25: saved metrics_gcn_h64_l2_hr80_c2p100_sim10_nonorm_epoch025.json
[train] epoch  26/120  loss=2.0168  time=7.4s
[train] epoch  27/120  loss=2.5594  time=7.4s
[train] epoch  28/120  loss=2.7164  time=7.4s
[train] epoch  29/120  loss=3.1964  time=7.4s
[train] epoch  30/120  loss=2.1234  time=7.4s
[ckpt] epoch 30: saved project_z_gcn_h64_l2_hr80_c2p100_sim10_nonorm_epoch030.npy, company_z_gcn_h64_l2_hr80_c2p100_sim10_nonorm_epoch030.npy
[ckpt] epoch 30: saved model_gcn_h64_l2_hr80_c2p100_sim10_nonorm_epoch030.pt

=== GNN [gcn] epoch 30 [p2c] K=100 ===
  royalty       recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0000  ndcg@100=0.0000
  commercial    recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0003  ndcg@100=0.0001
  performance   recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0005  ndcg@100=0.0001

=== GNN [gcn] epoch 30 [c2p] K=100 ===
  royalty       recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0000  ndcg@100=0.0000
  commercial    recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0001  ndcg@100=0.0000
  performance   recall@10=0.0003  ndcg@10=0.0001  recall@100=0.0012  ndcg@100=0.0003
[ckpt] epoch 30: saved metrics_gcn_h64_l2_hr80_c2p100_sim10_nonorm_epoch030.json
[train] epoch  31/120  loss=2.6060  time=7.5s
[train] epoch  32/120  loss=1.8451  time=7.4s
[train] epoch  33/120  loss=1.9223  time=7.4s
[train] epoch  34/120  loss=1.7003  time=7.4s
[train] epoch  35/120  loss=2.0043  time=7.4s
[ckpt] epoch 35: saved project_z_gcn_h64_l2_hr80_c2p100_sim10_nonorm_epoch035.npy, company_z_gcn_h64_l2_hr80_c2p100_sim10_nonorm_epoch035.npy
[ckpt] epoch 35: saved model_gcn_h64_l2_hr80_c2p100_sim10_nonorm_epoch035.pt

=== GNN [gcn] epoch 35 [p2c] K=100 ===
  royalty       recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0000  ndcg@100=0.0000
  commercial    recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0003  ndcg@100=0.0000
  performance   recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0002  ndcg@100=0.0000

=== GNN [gcn] epoch 35 [c2p] K=100 ===
  royalty       recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0000  ndcg@100=0.0000
  commercial    recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0001  ndcg@100=0.0000
  performance   recall@10=0.0006  ndcg@10=0.0002  recall@100=0.0013  ndcg@100=0.0003
[ckpt] epoch 35: saved metrics_gcn_h64_l2_hr80_c2p100_sim10_nonorm_epoch035.json
[train] epoch  36/120  loss=1.4815  time=7.5s
[train] epoch  37/120  loss=1.6999  time=7.4s
[train] epoch  38/120  loss=1.5435  time=7.4s
[train] epoch  39/120  loss=1.2743  time=7.4s
[train] epoch  40/120  loss=1.3311  time=7.4s
[ckpt] epoch 40: saved project_z_gcn_h64_l2_hr80_c2p100_sim10_nonorm_epoch040.npy, company_z_gcn_h64_l2_hr80_c2p100_sim10_nonorm_epoch040.npy
[ckpt] epoch 40: saved model_gcn_h64_l2_hr80_c2p100_sim10_nonorm_epoch040.pt

=== GNN [gcn] epoch 40 [p2c] K=100 ===
  royalty       recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0000  ndcg@100=0.0000
  commercial    recall@10=0.0003  ndcg@10=0.0001  recall@100=0.0003  ndcg@100=0.0001
  performance   recall@10=0.0002  ndcg@10=0.0001  recall@100=0.0002  ndcg@100=0.0001

=== GNN [gcn] epoch 40 [c2p] K=100 ===
  royalty       recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0000  ndcg@100=0.0000
  commercial    recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0004  ndcg@100=0.0001
  performance   recall@10=0.0006  ndcg@10=0.0002  recall@100=0.0012  ndcg@100=0.0003
[ckpt] epoch 40: saved metrics_gcn_h64_l2_hr80_c2p100_sim10_nonorm_epoch040.json
[train] epoch  41/120  loss=1.9307  time=7.4s
[train] epoch  42/120  loss=1.3176  time=7.4s
[train] epoch  43/120  loss=1.3487  time=7.4s
[train] epoch  44/120  loss=1.2908  time=7.4s
[train] epoch  45/120  loss=1.2234  time=7.4s
[ckpt] epoch 45: saved project_z_gcn_h64_l2_hr80_c2p100_sim10_nonorm_epoch045.npy, company_z_gcn_h64_l2_hr80_c2p100_sim10_nonorm_epoch045.npy
[ckpt] epoch 45: saved model_gcn_h64_l2_hr80_c2p100_sim10_nonorm_epoch045.pt

=== GNN [gcn] epoch 45 [p2c] K=100 ===
  royalty       recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0000  ndcg@100=0.0000
  commercial    recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0003  ndcg@100=0.0001
  performance   recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0002  ndcg@100=0.0001

=== GNN [gcn] epoch 45 [c2p] K=100 ===
  royalty       recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0000  ndcg@100=0.0000
  commercial    recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0004  ndcg@100=0.0001
  performance   recall@10=0.0006  ndcg@10=0.0002  recall@100=0.0009  ndcg@100=0.0003
[ckpt] epoch 45: saved metrics_gcn_h64_l2_hr80_c2p100_sim10_nonorm_epoch045.json
[train] epoch  46/120  loss=1.3454  time=7.4s
[train] epoch  47/120  loss=1.0648  time=7.4s
[train] epoch  48/120  loss=1.1578  time=7.4s
[train] epoch  49/120  loss=1.2155  time=7.4s
[train] epoch  50/120  loss=1.0719  time=7.4s
[ckpt] epoch 50: saved project_z_gcn_h64_l2_hr80_c2p100_sim10_nonorm_epoch050.npy, company_z_gcn_h64_l2_hr80_c2p100_sim10_nonorm_epoch050.npy
[ckpt] epoch 50: saved model_gcn_h64_l2_hr80_c2p100_sim10_nonorm_epoch050.pt

=== GNN [gcn] epoch 50 [p2c] K=100 ===
  royalty       recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0000  ndcg@100=0.0000
  commercial    recall@10=0.0003  ndcg@10=0.0001  recall@100=0.0003  ndcg@100=0.0001
  performance   recall@10=0.0002  ndcg@10=0.0001  recall@100=0.0002  ndcg@100=0.0001

=== GNN [gcn] epoch 50 [c2p] K=100 ===
  royalty       recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0000  ndcg@100=0.0000
  commercial    recall@10=0.0000  ndcg@10=0.0000  recall@100=0.0004  ndcg@100=0.0001
  performance   recall@10=0.0003  ndcg@10=0.0001  recall@100=0.0010  ndcg@100=0.0003

In [ ]:
!git pull origin main && python scripts/train_gnn.py --graph-path /content/drive/MyDrive/apollo.M1.GNN/graph.pt --held-out-path data/processed/held_out.pt --with-similarity --sim-path data/processed/sim_edges_top20.npz --hard-neg-path data/processed/hard_negatives_p2c.npz --hard-neg-path-c2p data/processed/hard_negatives_c2p.npz --p2c-weight 1.0 --c2p-weight 1.0 --hard-ratio 0.8 --num-neg 20 --no-normalize --layer-type hgt --hidden-dim 64 --num-layers 1 --num-heads 2 --batch-size 1024 --epochs 120 --device cuda --checkpoint-every 5 --save-model-ckpt --eval-at-checkpoint --direction both --save-metrics data/processed/checkpoints/gnn_metrics_hr08.json --checkpoint-dir data/processed/checkpoints

From https://github.com/osung/apollo.M1.GNN
 * branch            main       -> FETCH_HEAD
Already up to date.
[train_gnn] loading graph /content/drive/MyDrive/apollo.M1.GNN/graph.pt
[train_gnn] loading similarity edges data/processed/sim_edges_top20.npz
[train_gnn] added similarity edges: |E|=24,188,099  weight=0.25  reverse=True
[train_gnn] layer_type=hgt  params=214,468  normalize_output=False
[train_gnn] loading p2c hard negatives data/processed/hard_negatives_p2c.npz
[train_gnn]   p2c: 430,374 projects  (min=20, median=20, max=20)
[train_gnn] loading c2p hard negatives data/processed/hard_negatives_c2p.npz
[train_gnn]   c2p: 817,117 companies  (min=20, median=20, max=20)
[train_gnn] epochs=120 batch_size=2048 num_neg=20 (p2c weight=1.0, n_hard=16, n_random=4; c2p enabled=True, weight=1.0, n_hard=16, n_random=4) lr=0.001 wd=1e-05 device=cuda
[train_gnn] experiment tag: hgt_h64_l2_nh4_hr80_c2p100_sim20_nonorm
[train_gnn] checkpointing every 5 epochs -> data/processed/checkpoints
Trac

In [ ]:
!git pull origin main && python scripts/train_gnn.py --graph-path /content/drive/MyDrive/apollo.M1.GNN/graph.pt --held-out-path data/processed/held_out.pt --with-similarity --sim-path data/processed/sim_edges_top10.npz --hard-neg-path data/processed/hard_negatives_p2c.npz --hard-neg-path-c2p data/processed/hard_negatives_c2p.npz --p2c-weight 1.0 --c2p-weight 1.0 --hard-ratio 0.8 --num-neg 20 --no-normalize --layer-type sage --hidden-dim 256 --num-layers 2 --num-heads 4 --epochs 120 --device cuda --checkpoint-every 5 --save-model-ckpt --eval-at-checkpoint --direction both --save-metrics data/processed/checkpoints/gnn_metrics_hr08.json --checkpoint-dir data/processed/checkpoints

remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 7 (delta 6), reused 7 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 1.88 KiB | 960.00 KiB/s, done.
From https://github.com/osung/apollo.M1.GNN
 * branch            main       -> FETCH_HEAD
   6448b14..3db3ab7  main       -> origin/main
Updating 6448b14..3db3ab7
Fast-forward
 scripts/train_gnn.py    | 12 +++++++++
 src/training/trainer.py | 72 ++++++++++++++++++++++++++++++++++---------------
 2 files changed, 62 insertions(+), 22 deletions(-)
[train_gnn] loading graph /content/drive/MyDrive/apollo.M1.GNN/graph.pt
[train_gnn] loading similarity edges data/processed/sim_edges_top10.npz
[train_gnn] added similarity edges: |E|=12,138,565  weight=0.25  reverse=True
[train_gnn] layer_type=sage  params=2,560,768  normalize_output=False
[train_gnn] loading p2c hard negatives data/processed/hard_negatives_p2c.npz
[train_gnn]   p2c: 

In [ ]:
!git pull origin main && python scripts/train_gnn.py --graph-path /content/drive/MyDrive/apollo.M1.GNN/graph.pt --held-out-path data/processed/held_out.pt --with-similarity --sim-path data/processed/sim_edges_top10.npz --hard-neg-path data/processed/hard_negatives_p2c.npz --hard-neg-path-c2p data/processed/hard_negatives_c2p.npz --p2c-weight 1.0 --c2p-weight 1.0 --hard-ratio 0.8 --num-neg 20 --no-normalize --layer-type hgt --hidden-dim 96 --num-heads 2 --num-layers 2 --amp-dtype bf16 --epochs 120 --device cuda --checkpoint-every 5 --save-model-ckpt --eval-at-checkpoint --direction both --checkpoint-dir data/processed/checkpoints

remote: Enumerating objects: 19, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 13 (delta 10), reused 9 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (13/13), 3.08 KiB | 789.00 KiB/s, done.
From https://github.com/osung/apollo.M1.GNN
 * branch            main       -> FETCH_HEAD
   3db3ab7..b26c7c0  main       -> origin/main
Updating 3db3ab7..b26c7c0
Fast-forward
 scripts/build_hard_negatives.py | 36 +++++++++++++++++++++++++++++
 src/training/trainer.py         | 50 ++++++++++++++++++++++++-----------------
 2 files changed, 66 insertions(+), 20 deletions(-)
[train_gnn] loading graph /content/drive/MyDrive/apollo.M1.GNN/graph.pt
[train_gnn] loading similarity edges data/processed/sim_edges_top10.npz
[train_gnn] added similarity edges: |E|=12,138,565  weight=0.25  reverse=True
[train_gnn] layer_type=hgt  params=468,964  normalize_output=False
[train_gnn] loading p2c hard negatives data/processed/hard_negative

In [ ]:
!git pull origin main && python scripts/train_gnn.py --graph-path /content/drive/MyDrive/apollo.M1.GNN/graph.pt --held-out-path data/processed/held_out.pt --with-similarity --sim-path data/processed/sim_edges_top10.npz --hard-neg-path data/processed/hard_negatives_p2c_clean.npz --hard-neg-path-c2p data/processed/hard_negatives_c2p_clean.npz --p2c-weight 1.0 --c2p-weight 1.0 --hard-ratio 0.8 --num-neg 20 --no-normalize --layer-type sage --hidden-dim 256 --num-layers 2 --epochs 150 --device cuda --checkpoint-every 5 --save-model-ckpt --eval-at-checkpoint --direction both --checkpoint-dir /content/drive/MyDrive/apollo.M1.GNN/outputs --save-metrics /content/drive/MyDrive/apollo.M1.GNN/outputs/metrics_sage_h256_clean.json

From https://github.com/osung/apollo.M1.GNN
 * branch            main       -> FETCH_HEAD
Already up to date.
[train_gnn] loading graph /content/drive/MyDrive/apollo.M1.GNN/graph.pt
[train_gnn] loading similarity edges data/processed/sim_edges_top10.npz
[train_gnn] added similarity edges: |E|=12,138,565  weight=0.25  reverse=True
[train_gnn] layer_type=sage  params=2,560,768  normalize_output=False
[train_gnn] loading p2c hard negatives data/processed/hard_negatives_p2c_clean.npz
[train_gnn]   p2c: 430,370 projects  (min=1, median=20, max=20)
[train_gnn] loading c2p hard negatives data/processed/hard_negatives_c2p_clean.npz
[train_gnn]   c2p: 817,117 companies  (min=20, median=20, max=20)
[train_gnn] epochs=150 batch_size=4096 num_neg=20 (p2c weight=1.0, n_hard=16, n_random=4; c2p enabled=True, weight=1.0, n_hard=16, n_random=4) lr=0.001 wd=1e-05 device=cuda
[train_gnn] experiment tag: sage_h256_l2_hr80_c2p100_sim10_nonorm
[train_gnn] checkpointing every 5 epochs -> /content/drive/MyDr

In [ ]:
!git pull origin main && python scripts/train_gnn.py --graph-path /content/drive/MyDrive/apollo.M1.GNN/graph.pt --held-out-path data/processed/held_out.pt --with-similarity --sim-path data/processed/sim_edges_top10.npz --hard-neg-path data/processed/hard_negatives_p2c_clean.npz --hard-neg-path-c2p data/processed/hard_negatives_c2p_clean.npz --p2c-weight 1.0 --c2p-weight 1.0 --hard-ratio 0.8 --num-neg 20 --layer-type gcn --hidden-dim 64 --num-layers 2 --epochs 150 --device cuda --checkpoint-every 5 --save-model-ckpt --eval-at-checkpoint --direction both --checkpoint-dir /content/drive/MyDrive/apollo.M1.GNN/outputs --save-metrics /content/drive/MyDrive/apollo.M1.GNN/outputs/metrics_sage_h256_clean.json

From https://github.com/osung/apollo.M1.GNN
 * branch            main       -> FETCH_HEAD
Already up to date.
[train_gnn] loading graph /content/drive/MyDrive/apollo.M1.GNN/graph.pt
[train_gnn] loading similarity edges data/processed/sim_edges_top10.npz
[train_gnn] added similarity edges: |E|=12,138,565  weight=0.25  reverse=True
[train_gnn] layer_type=gcn  params=247,168  normalize_output=True
[train_gnn] loading p2c hard negatives data/processed/hard_negatives_p2c_clean.npz
[train_gnn]   p2c: 430,370 projects  (min=1, median=20, max=20)
[train_gnn] loading c2p hard negatives data/processed/hard_negatives_c2p_clean.npz
[train_gnn]   c2p: 817,117 companies  (min=20, median=20, max=20)
[train_gnn] epochs=150 batch_size=4096 num_neg=20 (p2c weight=1.0, n_hard=16, n_random=4; c2p enabled=True, weight=1.0, n_hard=16, n_random=4) lr=0.001 wd=1e-05 device=cuda
[train_gnn] experiment tag: gcn_h64_l2_hr80_c2p100_sim10
[train_gnn] checkpointing every 5 epochs -> /content/drive/MyDrive/apollo.M1

In [ ]:
!git pull origin main && python scripts/train_gnn.py --graph-path /content/drive/MyDrive/apollo.M1.GNN/graph.pt --held-out-path data/processed/held_out.pt --with-similarity --sim-path data/processed/sim_edges_top10.npz --hard-neg-path data/processed/hard_negatives_p2c_clean.npz --hard-neg-path-c2p data/processed/hard_negatives_c2p_clean.npz --p2c-weight 1.0 --c2p-weight 1.0 --hard-ratio 0.8 --num-neg 20 --layer-type gcn --hidden-dim 64 --num-layers 2 --epochs 150 --device cuda --checkpoint-every 5 --save-model-ckpt --eval-at-checkpoint --direction both --no-normalize --checkpoint-dir /content/drive/MyDrive/apollo.M1.GNN/outputs --save-metrics /content/drive/MyDrive/apollo.M1.GNN/outputs/metrics_sage_h256_clean.json

From https://github.com/osung/apollo.M1.GNN
 * branch            main       -> FETCH_HEAD
Already up to date.
[train_gnn] loading graph /content/drive/MyDrive/apollo.M1.GNN/graph.pt
[train_gnn] loading similarity edges data/processed/sim_edges_top10.npz
[train_gnn] added similarity edges: |E|=12,138,565  weight=0.25  reverse=True
[train_gnn] layer_type=gcn  params=247,168  normalize_output=False
[train_gnn] loading p2c hard negatives data/processed/hard_negatives_p2c_clean.npz
[train_gnn]   p2c: 430,370 projects  (min=1, median=20, max=20)
[train_gnn] loading c2p hard negatives data/processed/hard_negatives_c2p_clean.npz
[train_gnn]   c2p: 817,117 companies  (min=20, median=20, max=20)
[train_gnn] epochs=150 batch_size=4096 num_neg=20 (p2c weight=1.0, n_hard=16, n_random=4; c2p enabled=True, weight=1.0, n_hard=16, n_random=4) lr=0.001 wd=1e-05 device=cuda
[train_gnn] experiment tag: gcn_h64_l2_hr80_c2p100_sim10_nonorm
[train_gnn] checkpointing every 5 epochs -> /content/drive/MyDrive/a

In [ ]:
!git pull origin main && python scripts/train_gnn.py --graph-path /content/drive/MyDrive/apollo.M1.GNN/graph.pt --held-out-path data/processed/held_out.pt --with-similarity --sim-path data/processed/sim_edges_top10.npz --hard-neg-path data/processed/hard_negatives_p2c_clean.npz --hard-neg-path-c2p data/processed/hard_negatives_c2p_clean.npz --p2c-weight 1.0 --c2p-weight 1.0 --hard-ratio 0.8 --num-neg 20 --layer-type gcn --hidden-dim 128 --num-layers 2 --epochs 150 --device cuda --checkpoint-every 5 --save-model-ckpt --eval-at-checkpoint --direction both --no-normalize --checkpoint-dir /content/drive/MyDrive/apollo.M1.GNN/outputs --save-metrics /content/drive/MyDrive/apollo.M1.GNN/outputs/metrics_sage_h256_clean.json

From https://github.com/osung/apollo.M1.GNN
 * branch            main       -> FETCH_HEAD
Already up to date.
[train_gnn] loading graph /content/drive/MyDrive/apollo.M1.GNN/graph.pt
[train_gnn] loading similarity edges data/processed/sim_edges_top10.npz
[train_gnn] added similarity edges: |E|=12,138,565  weight=0.25  reverse=True
[train_gnn] layer_type=gcn  params=756,224  normalize_output=False
[train_gnn] loading p2c hard negatives data/processed/hard_negatives_p2c_clean.npz
[train_gnn]   p2c: 430,370 projects  (min=1, median=20, max=20)
[train_gnn] loading c2p hard negatives data/processed/hard_negatives_c2p_clean.npz
[train_gnn]   c2p: 817,117 companies  (min=20, median=20, max=20)
[train_gnn] epochs=150 batch_size=4096 num_neg=20 (p2c weight=1.0, n_hard=16, n_random=4; c2p enabled=True, weight=1.0, n_hard=16, n_random=4) lr=0.001 wd=1e-05 device=cuda
[train_gnn] experiment tag: gcn_h128_l2_hr80_c2p100_sim10_nonorm
[train_gnn] checkpointing every 5 epochs -> /content/drive/MyDrive/

In [ ]:
!git pull origin main && python scripts/train_gnn.py --graph-path /content/drive/MyDrive/apollo.M1.GNN/graph.pt --held-out-path data/processed/held_out.pt --with-similarity --sim-path data/processed/sim_edges_top10.npz --hard-neg-path data/processed/hard_negatives_p2c_clean.npz --hard-neg-path-c2p data/processed/hard_negatives_c2p_clean.npz --p2c-weight 1.0 --c2p-weight 1.0 --hard-ratio 0.8 --num-neg 20 --no-normalize --layer-type gcn --hidden-dim 128 --num-layers 2 --epochs 200 --device cuda --checkpoint-every 5 --save-model-ckpt --eval-at-checkpoint --direction both --checkpoint-dir /content/drive/MyDrive/apollo.M1.GNN/outputs --resume /content/drive/MyDrive/apollo.M1.GNN/outputs/model_gcn_h128_l2_hr80_c2p100_sim10_nonorm_epoch145.pt

remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 7 (delta 6), reused 7 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 1.27 KiB | 652.00 KiB/s, done.
From https://github.com/osung/apollo.M1.GNN
 * branch            main       -> FETCH_HEAD
   b26c7c0..eac03d7  main       -> origin/main
Updating b26c7c0..eac03d7
Fast-forward
 scripts/train_gnn.py    | 27 +++++++++++++++++++++++++++
 src/training/trainer.py |  1 -
 2 files changed, 27 insertions(+), 1 deletion(-)
[train_gnn] loading graph /content/drive/MyDrive/apollo.M1.GNN/graph.pt
[train_gnn] loading similarity edges data/processed/sim_edges_top10.npz
[train_gnn] added similarity edges: |E|=12,138,565  weight=0.25  reverse=True
[train_gnn] layer_type=gcn  params=756,224  normalize_output=False
[train_gnn] loading p2c hard negatives data/processed/hard_negatives_p2c_clean.npz
[train_gnn]   p2c: 430,370 projects  (min=1, med

In [ ]:
!git pull origin main && python scripts/train_gnn.py --graph-path /content/drive/MyDrive/apollo.M1.GNN/graph.pt --held-out-path data/processed/held_out.pt --with-similarity --sim-path data/processed/sim_edges_top10.npz --hard-neg-path data/processed/hard_negatives_p2c_clean.npz --hard-neg-path-c2p data/processed/hard_negatives_c2p_clean.npz --p2c-weight 1.0 --c2p-weight 1.0 --hard-ratio 0.8 --num-neg 20 --no-normalize --layer-type gcn --hidden-dim 384 --num-layers 2 --epochs 200 --device cuda --checkpoint-every 5 --save-model-ckpt --eval-at-checkpoint --direction both --checkpoint-dir /content/drive/MyDrive/apollo.M1.GNN/outputs

From https://github.com/osung/apollo.M1.GNN
 * branch            main       -> FETCH_HEAD
Already up to date.
[train_gnn] loading graph /content/drive/MyDrive/apollo.M1.GNN/graph.pt
[train_gnn] loading similarity edges data/processed/sim_edges_top10.npz
[train_gnn] added similarity edges: |E|=12,138,565  weight=0.25  reverse=True
[train_gnn] layer_type=gcn  params=5,413,888  normalize_output=False
[train_gnn] loading p2c hard negatives data/processed/hard_negatives_p2c_clean.npz
[train_gnn]   p2c: 430,370 projects  (min=1, median=20, max=20)
[train_gnn] loading c2p hard negatives data/processed/hard_negatives_c2p_clean.npz
[train_gnn]   c2p: 817,117 companies  (min=20, median=20, max=20)
[train_gnn] epochs=200 batch_size=4096 num_neg=20 (p2c weight=1.0, n_hard=16, n_random=4; c2p enabled=True, weight=1.0, n_hard=16, n_random=4) lr=0.001 wd=1e-05 device=cuda
[train_gnn] experiment tag: gcn_h384_l2_hr80_c2p100_sim10_nonorm
[train_gnn] checkpointing every 5 epochs -> /content/drive/MyDriv

In [ ]:
!git pull origin main && python scripts/train_gnn.py --graph-path /content/drive/MyDrive/apollo.M1.GNN/graph.pt --held-out-path data/processed/held_out.pt --with-similarity --sim-path data/processed/sim_edges_top10.npz --hard-neg-path data/processed/hard_negatives_p2c_clean.npz --hard-neg-path-c2p data/processed/hard_negatives_c2p_clean.npz --p2c-weight 1.0 --c2p-weight 1.0 --hard-ratio 0.9 --num-neg 20 --no-normalize --layer-type gcn --hidden-dim 384 --num-layers 2 --epochs 200 --device cuda --checkpoint-every 5 --save-model-ckpt --eval-at-checkpoint --direction both --checkpoint-dir /content/drive/MyDrive/apollo.M1.GNN/outputs

remote: Enumerating objects: 27, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 22 (delta 16), reused 14 (delta 8), pack-reused 0 (from 0)
Unpacking objects: 100% (22/22), 14.89 KiB | 2.13 MiB/s, done.
From https://github.com/osung/apollo.M1.GNN
 * branch            main       -> FETCH_HEAD
   eac03d7..6034887  main       -> origin/main
Updating eac03d7..6034887
Fast-forward
 scripts/run_sweep.py   | 501 +++++++++++++++++++++++++++++++++++++++++++++++++
 scripts/train_gnn.py   |  44 +++--
 src/models/lightgcn.py | 145 ++++++++++++++
 src/models/sehgnn.py   | 187 ++++++++++++++++++
 4 files changed, 866 insertions(+), 11 deletions(-)
 create mode 100644 scripts/run_sweep.py
 create mode 100644 src/models/lightgcn.py
 create mode 100644 src/models/sehgnn.py
[train_gnn] loading graph /content/drive/MyDrive/apollo.M1.GNN/graph.pt
[train_gnn] loading similarity edges data/processed/sim_edges_top10.npz
[train_gnn] added simi

In [ ]:
=== GNN [gcn] epoch 85 [p2c] K=100 ===
  royalty       recall@10=0.1444  ndcg@10=0.0822  recall@100=0.3724  ndcg@100=0.1293
  commercial    recall@10=0.1308  ndcg@10=0.0687  recall@100=0.3354  ndcg@100=0.1092
  performance   recall@10=0.1015  ndcg@10=0.0550  recall@100=0.2758  ndcg@100=0.0905

=== GNN [gcn] epoch 85 [c2p] K=100 ===
  royalty       recall@10=0.0478  ndcg@10=0.0242  recall@100=0.2135  ndcg@100=0.0566
  commercial    recall@10=0.0422  ndcg@10=0.0222  recall@100=0.1769  ndcg@100=0.0491
  performance   recall@10=0.0387  ndcg@10=0.0187  recall@100=0.1483  ndcg@100=0.0412
[ckpt] epoch 85: saved metrics_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch085.json
[train] epoch  86/200  loss=0.0015  time=41.9s
[train] epoch  87/200  loss=0.0010  time=41.9s
[train] epoch  88/200  loss=0.0010  time=41.8s
[train] epoch  89/200  loss=0.0011  time=41.8s
[train] epoch  90/200  loss=0.0009  time=41.8s
[ckpt] epoch 90: saved project_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch090.npy, company_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch090.npy
[ckpt] epoch 90: saved model_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch090.pt

=== GNN [gcn] epoch 90 [p2c] K=100 ===
  royalty       recall@10=0.2280  ndcg@10=0.1341  recall@100=0.4378  ndcg@100=0.1774
  commercial    recall@10=0.1881  ndcg@10=0.1062  recall@100=0.4092  ndcg@100=0.1523
  performance   recall@10=0.1696  ndcg@10=0.0989  recall@100=0.3495  ndcg@100=0.1364

=== GNN [gcn] epoch 90 [c2p] K=100 ===
  royalty       recall@10=0.0794  ndcg@10=0.0434  recall@100=0.2723  ndcg@100=0.0811
  commercial    recall@10=0.0416  ndcg@10=0.0213  recall@100=0.1817  ndcg@100=0.0488
  performance   recall@10=0.0425  ndcg@10=0.0207  recall@100=0.1972  ndcg@100=0.0518
[ckpt] epoch 90: saved metrics_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch090.json
[train] epoch  91/200  loss=0.0011  time=41.9s
[train] epoch  92/200  loss=0.0009  time=41.9s
[train] epoch  93/200  loss=0.0009  time=41.9s
[train] epoch  94/200  loss=0.0008  time=41.8s
[train] epoch  95/200  loss=0.0007  time=41.8s
[ckpt] epoch 95: saved project_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch095.npy, company_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch095.npy
[ckpt] epoch 95: saved model_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch095.pt

=== GNN [gcn] epoch 95 [p2c] K=100 ===
  royalty       recall@10=0.2280  ndcg@10=0.1378  recall@100=0.4310  ndcg@100=0.1796
  commercial    recall@10=0.2011  ndcg@10=0.1140  recall@100=0.4027  ndcg@100=0.1563
  performance   recall@10=0.1667  ndcg@10=0.0975  recall@100=0.3304  ndcg@100=0.1311

=== GNN [gcn] epoch 95 [c2p] K=100 ===
  royalty       recall@10=0.0815  ndcg@10=0.0437  recall@100=0.2722  ndcg@100=0.0818
  commercial    recall@10=0.0617  ndcg@10=0.0317  recall@100=0.2323  ndcg@100=0.0658
  performance   recall@10=0.0590  ndcg@10=0.0301  recall@100=0.2282  ndcg@100=0.0642
[ckpt] epoch 95: saved metrics_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch095.json
[train] epoch  96/200  loss=0.0009  time=41.9s
[train] epoch  97/200  loss=0.0010  time=41.8s
[train] epoch  98/200  loss=0.0008  time=41.8s
[train] epoch  99/200  loss=0.0008  time=41.8s
[train] epoch 100/200  loss=0.0007  time=41.8s
[ckpt] epoch 100: saved project_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch100.npy, company_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch100.npy
[ckpt] epoch 100: saved model_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch100.pt

=== GNN [gcn] epoch 100 [p2c] K=100 ===
  royalty       recall@10=0.2673  ndcg@10=0.1570  recall@100=0.4467  ndcg@100=0.1947
  commercial    recall@10=0.2224  ndcg@10=0.1323  recall@100=0.4087  ndcg@100=0.1718
  performance   recall@10=0.1808  ndcg@10=0.1059  recall@100=0.3425  ndcg@100=0.1396

=== GNN [gcn] epoch 100 [c2p] K=100 ===
  royalty       recall@10=0.1078  ndcg@10=0.0614  recall@100=0.2988  ndcg@100=0.1002
  commercial    recall@10=0.0582  ndcg@10=0.0307  recall@100=0.2369  ndcg@100=0.0667
  performance   recall@10=0.0556  ndcg@10=0.0292  recall@100=0.2120  ndcg@100=0.0610
[ckpt] epoch 100: saved metrics_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch100.json
[train] epoch 101/200  loss=0.0009  time=41.9s
[train] epoch 102/200  loss=0.0009  time=41.8s
[train] epoch 103/200  loss=0.0010  time=41.8s
[train] epoch 104/200  loss=0.0009  time=41.8s
[train] epoch 105/200  loss=0.0010  time=41.8s
[ckpt] epoch 105: saved project_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch105.npy, company_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch105.npy
[ckpt] epoch 105: saved model_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch105.pt

=== GNN [gcn] epoch 105 [p2c] K=100 ===
  royalty       recall@10=0.1647  ndcg@10=0.0967  recall@100=0.3808  ndcg@100=0.1404
  commercial    recall@10=0.1312  ndcg@10=0.0744  recall@100=0.3189  ndcg@100=0.1121
  performance   recall@10=0.1139  ndcg@10=0.0640  recall@100=0.2797  ndcg@100=0.0976

=== GNN [gcn] epoch 105 [c2p] K=100 ===
  royalty       recall@10=0.0457  ndcg@10=0.0218  recall@100=0.2129  ndcg@100=0.0542
  commercial    recall@10=0.0317  ndcg@10=0.0163  recall@100=0.1651  ndcg@100=0.0424
  performance   recall@10=0.0276  ndcg@10=0.0134  recall@100=0.1350  ndcg@100=0.0350
[ckpt] epoch 105: saved metrics_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch105.json
[train] epoch 106/200  loss=0.0030  time=41.9s
[train] epoch 107/200  loss=0.0031  time=41.8s
[train] epoch 108/200  loss=0.0017  time=41.8s
[train] epoch 109/200  loss=0.0012  time=41.8s
[train] epoch 110/200  loss=0.0011  time=41.8s
[ckpt] epoch 110: saved project_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch110.npy, company_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch110.npy
[ckpt] epoch 110: saved model_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch110.pt

=== GNN [gcn] epoch 110 [p2c] K=100 ===
  royalty       recall@10=0.1872  ndcg@10=0.1059  recall@100=0.4215  ndcg@100=0.1556
  commercial    recall@10=0.1520  ndcg@10=0.0833  recall@100=0.3745  ndcg@100=0.1287
  performance   recall@10=0.1475  ndcg@10=0.0820  recall@100=0.3166  ndcg@100=0.1171

=== GNN [gcn] epoch 110 [c2p] K=100 ===
  royalty       recall@10=0.0741  ndcg@10=0.0419  recall@100=0.2653  ndcg@100=0.0792
  commercial    recall@10=0.0459  ndcg@10=0.0250  recall@100=0.2081  ndcg@100=0.0570
  performance   recall@10=0.0357  ndcg@10=0.0185  recall@100=0.1769  ndcg@100=0.0468
[ckpt] epoch 110: saved metrics_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch110.json
[train] epoch 111/200  loss=0.0008  time=41.9s
[train] epoch 112/200  loss=0.0015  time=41.8s
[train] epoch 113/200  loss=0.0013  time=41.8s
[train] epoch 114/200  loss=0.0010  time=41.8s
[train] epoch 115/200  loss=0.0014  time=41.8s
[ckpt] epoch 115: saved project_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch115.npy, company_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch115.npy
[ckpt] epoch 115: saved model_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch115.pt

=== GNN [gcn] epoch 115 [p2c] K=100 ===
  royalty       recall@10=0.2008  ndcg@10=0.1195  recall@100=0.4294  ndcg@100=0.1663
  commercial    recall@10=0.1930  ndcg@10=0.1102  recall@100=0.3914  ndcg@100=0.1513
  performance   recall@10=0.1505  ndcg@10=0.0846  recall@100=0.3285  ndcg@100=0.1219

=== GNN [gcn] epoch 115 [c2p] K=100 ===
  royalty       recall@10=0.0757  ndcg@10=0.0389  recall@100=0.2713  ndcg@100=0.0776
  commercial    recall@10=0.0477  ndcg@10=0.0246  recall@100=0.2042  ndcg@100=0.0558
  performance   recall@10=0.0466  ndcg@10=0.0250  recall@100=0.1881  ndcg@100=0.0539
[ckpt] epoch 115: saved metrics_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch115.json
[train] epoch 116/200  loss=0.0013  time=41.9s
[train] epoch 117/200  loss=0.0010  time=41.8s
[train] epoch 118/200  loss=0.0011  time=41.8s
[train] epoch 119/200  loss=0.0013  time=41.8s
[train] epoch 120/200  loss=0.0011  time=41.8s
[ckpt] epoch 120: saved project_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch120.npy, company_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch120.npy
[ckpt] epoch 120: saved model_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch120.pt

=== GNN [gcn] epoch 120 [p2c] K=100 ===
  royalty       recall@10=0.2490  ndcg@10=0.1469  recall@100=0.4440  ndcg@100=0.1896
  commercial    recall@10=0.2158  ndcg@10=0.1235  recall@100=0.4052  ndcg@100=0.1633
  performance   recall@10=0.1994  ndcg@10=0.1179  recall@100=0.3511  ndcg@100=0.1501

=== GNN [gcn] epoch 120 [c2p] K=100 ===
  royalty       recall@10=0.0573  ndcg@10=0.0317  recall@100=0.2440  ndcg@100=0.0688
  commercial    recall@10=0.0351  ndcg@10=0.0176  recall@100=0.1807  ndcg@100=0.0465
  performance   recall@10=0.0462  ndcg@10=0.0246  recall@100=0.1913  ndcg@100=0.0537
[ckpt] epoch 120: saved metrics_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch120.json
[train] epoch 121/200  loss=0.0008  time=41.9s
[train] epoch 122/200  loss=0.0009  time=41.8s
[train] epoch 123/200  loss=0.0008  time=41.8s
[train] epoch 124/200  loss=0.0008  time=41.8s
[train] epoch 125/200  loss=0.0007  time=41.8s
[ckpt] epoch 125: saved project_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch125.npy, company_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch125.npy
[ckpt] epoch 125: saved model_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch125.pt

=== GNN [gcn] epoch 125 [p2c] K=100 ===
  royalty       recall@10=0.3117  ndcg@10=0.1911  recall@100=0.4571  ndcg@100=0.2231
  commercial    recall@10=0.2765  ndcg@10=0.1713  recall@100=0.4236  ndcg@100=0.2032
  performance   recall@10=0.2225  ndcg@10=0.1399  recall@100=0.3626  ndcg@100=0.1702

=== GNN [gcn] epoch 125 [c2p] K=100 ===
  royalty       recall@10=0.1036  ndcg@10=0.0539  recall@100=0.3074  ndcg@100=0.0953
  commercial    recall@10=0.0780  ndcg@10=0.0425  recall@100=0.2752  ndcg@100=0.0822
  performance   recall@10=0.0588  ndcg@10=0.0289  recall@100=0.2332  ndcg@100=0.0646
[ckpt] epoch 125: saved metrics_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch125.json
[train] epoch 126/200  loss=0.0008  time=41.9s
[train] epoch 127/200  loss=0.0006  time=41.8s
[train] epoch 128/200  loss=0.0006  time=41.8s
[train] epoch 129/200  loss=0.0007  time=41.8s
[train] epoch 130/200  loss=0.0009  time=41.8s
[ckpt] epoch 130: saved project_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch130.npy, company_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch130.npy
[ckpt] epoch 130: saved model_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch130.pt

=== GNN [gcn] epoch 130 [p2c] K=100 ===
  royalty       recall@10=0.2578  ndcg@10=0.1577  recall@100=0.4435  ndcg@100=0.1985
  commercial    recall@10=0.2447  ndcg@10=0.1457  recall@100=0.4090  ndcg@100=0.1806
  performance   recall@10=0.1929  ndcg@10=0.1139  recall@100=0.3456  ndcg@100=0.1462

=== GNN [gcn] epoch 130 [c2p] K=100 ===
  royalty       recall@10=0.0820  ndcg@10=0.0451  recall@100=0.2944  ndcg@100=0.0867
  commercial    recall@10=0.0541  ndcg@10=0.0296  recall@100=0.2308  ndcg@100=0.0649
  performance   recall@10=0.0498  ndcg@10=0.0258  recall@100=0.2210  ndcg@100=0.0604
[ckpt] epoch 130: saved metrics_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch130.json
[train] epoch 131/200  loss=0.0009  time=41.9s
[train] epoch 132/200  loss=0.0007  time=41.9s
[train] epoch 133/200  loss=0.0011  time=41.8s
[train] epoch 134/200  loss=0.0009  time=41.9s
[train] epoch 135/200  loss=0.0014  time=41.8s
[ckpt] epoch 135: saved project_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch135.npy, company_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch135.npy
[ckpt] epoch 135: saved model_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch135.pt

=== GNN [gcn] epoch 135 [p2c] K=100 ===
  royalty       recall@10=0.2259  ndcg@10=0.1286  recall@100=0.4419  ndcg@100=0.1730
  commercial    recall@10=0.2093  ndcg@10=0.1217  recall@100=0.4047  ndcg@100=0.1624
  performance   recall@10=0.1470  ndcg@10=0.0864  recall@100=0.3171  ndcg@100=0.1214

=== GNN [gcn] epoch 135 [c2p] K=100 ===
  royalty       recall@10=0.0652  ndcg@10=0.0322  recall@100=0.2434  ndcg@100=0.0680
  commercial    recall@10=0.0583  ndcg@10=0.0285  recall@100=0.2491  ndcg@100=0.0664
  performance   recall@10=0.0269  ndcg@10=0.0123  recall@100=0.1490  ndcg@100=0.0368
[ckpt] epoch 135: saved metrics_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch135.json
[train] epoch 136/200  loss=0.0013  time=41.9s
[train] epoch 137/200  loss=0.0009  time=41.9s
[train] epoch 138/200  loss=0.0010  time=41.9s
[train] epoch 139/200  loss=0.0009  time=41.8s
[train] epoch 140/200  loss=0.0007  time=41.8s
[ckpt] epoch 140: saved project_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch140.npy, company_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch140.npy
[ckpt] epoch 140: saved model_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch140.pt

=== GNN [gcn] epoch 140 [p2c] K=100 ===
  royalty       recall@10=0.2714  ndcg@10=0.1642  recall@100=0.4482  ndcg@100=0.2027
  commercial    recall@10=0.2452  ndcg@10=0.1459  recall@100=0.4175  ndcg@100=0.1830
  performance   recall@10=0.1929  ndcg@10=0.1178  recall@100=0.3523  ndcg@100=0.1516

=== GNN [gcn] epoch 140 [c2p] K=100 ===
  royalty       recall@10=0.1193  ndcg@10=0.0591  recall@100=0.3058  ndcg@100=0.0961
  commercial    recall@10=0.0615  ndcg@10=0.0310  recall@100=0.2632  ndcg@100=0.0719
  performance   recall@10=0.0544  ndcg@10=0.0276  recall@100=0.2115  ndcg@100=0.0594
[ckpt] epoch 140: saved metrics_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch140.json
[train] epoch 141/200  loss=0.0008  time=41.9s
[train] epoch 142/200  loss=0.0010  time=41.9s
[train] epoch 143/200  loss=0.0008  time=41.8s
[train] epoch 144/200  loss=0.0007  time=41.8s
[train] epoch 145/200  loss=0.0010  time=41.8s
[ckpt] epoch 145: saved project_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch145.npy, company_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch145.npy
[ckpt] epoch 145: saved model_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch145.pt

=== GNN [gcn] epoch 145 [p2c] K=100 ===
  royalty       recall@10=0.2369  ndcg@10=0.1407  recall@100=0.4430  ndcg@100=0.1847
  commercial    recall@10=0.2034  ndcg@10=0.1157  recall@100=0.4026  ndcg@100=0.1574
  performance   recall@10=0.1701  ndcg@10=0.0976  recall@100=0.3483  ndcg@100=0.1350

=== GNN [gcn] epoch 145 [c2p] K=100 ===
  royalty       recall@10=0.1078  ndcg@10=0.0571  recall@100=0.2855  ndcg@100=0.0934
  commercial    recall@10=0.0639  ndcg@10=0.0319  recall@100=0.2265  ndcg@100=0.0645
  performance   recall@10=0.0558  ndcg@10=0.0282  recall@100=0.2085  ndcg@100=0.0596
[ckpt] epoch 145: saved metrics_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch145.json
[train] epoch 146/200  loss=0.0009  time=41.9s
[train] epoch 147/200  loss=0.0009  time=41.8s
[train] epoch 148/200  loss=0.0007  time=41.8s
[train] epoch 149/200  loss=0.0007  time=41.8s
[train] epoch 150/200  loss=0.0032  time=41.8s
[ckpt] epoch 150: saved project_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch150.npy, company_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch150.npy
[ckpt] epoch 150: saved model_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch150.pt

=== GNN [gcn] epoch 150 [p2c] K=100 ===
  royalty       recall@10=0.1674  ndcg@10=0.0888  recall@100=0.4032  ndcg@100=0.1378
  commercial    recall@10=0.1572  ndcg@10=0.0899  recall@100=0.3674  ndcg@100=0.1331
  performance   recall@10=0.1077  ndcg@10=0.0602  recall@100=0.2928  ndcg@100=0.0976

=== GNN [gcn] epoch 150 [c2p] K=100 ===
  royalty       recall@10=0.0452  ndcg@10=0.0212  recall@100=0.1898  ndcg@100=0.0492
  commercial    recall@10=0.0256  ndcg@10=0.0133  recall@100=0.1527  ndcg@100=0.0380
  performance   recall@10=0.0279  ndcg@10=0.0126  recall@100=0.1321  ndcg@100=0.0334
[ckpt] epoch 150: saved metrics_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch150.json
[train] epoch 151/200  loss=0.0017  time=41.9s
[train] epoch 152/200  loss=0.0012  time=41.9s
[train] epoch 153/200  loss=0.0008  time=41.9s
[train] epoch 154/200  loss=0.0008  time=41.9s
[train] epoch 155/200  loss=0.0015  time=41.8s
[ckpt] epoch 155: saved project_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch155.npy, company_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch155.npy
[ckpt] epoch 155: saved model_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch155.pt

=== GNN [gcn] epoch 155 [p2c] K=100 ===
  royalty       recall@10=0.2558  ndcg@10=0.1525  recall@100=0.4472  ndcg@100=0.1938
  commercial    recall@10=0.2281  ndcg@10=0.1379  recall@100=0.4120  ndcg@100=0.1773
  performance   recall@10=0.1765  ndcg@10=0.1014  recall@100=0.3487  ndcg@100=0.1376

=== GNN [gcn] epoch 155 [c2p] K=100 ===
  royalty       recall@10=0.0794  ndcg@10=0.0427  recall@100=0.2580  ndcg@100=0.0773
  commercial    recall@10=0.0237  ndcg@10=0.0116  recall@100=0.1457  ndcg@100=0.0354
  performance   recall@10=0.0430  ndcg@10=0.0227  recall@100=0.2050  ndcg@100=0.0553
[ckpt] epoch 155: saved metrics_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch155.json
[train] epoch 156/200  loss=0.0011  time=41.9s
[train] epoch 157/200  loss=0.0045  time=41.8s
[train] epoch 158/200  loss=0.0024  time=41.8s
[train] epoch 159/200  loss=0.0018  time=41.8s
[train] epoch 160/200  loss=0.0014  time=41.8s
[ckpt] epoch 160: saved project_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch160.npy, company_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch160.npy
[ckpt] epoch 160: saved model_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch160.pt

=== GNN [gcn] epoch 160 [p2c] K=100 ===
  royalty       recall@10=0.1972  ndcg@10=0.1082  recall@100=0.4179  ndcg@100=0.1540
  commercial    recall@10=0.1872  ndcg@10=0.1054  recall@100=0.3867  ndcg@100=0.1469
  performance   recall@10=0.1429  ndcg@10=0.0796  recall@100=0.3135  ndcg@100=0.1151

=== GNN [gcn] epoch 160 [c2p] K=100 ===
  royalty       recall@10=0.0852  ndcg@10=0.0446  recall@100=0.2539  ndcg@100=0.0777
  commercial    recall@10=0.0522  ndcg@10=0.0279  recall@100=0.2063  ndcg@100=0.0590
  performance   recall@10=0.0520  ndcg@10=0.0268  recall@100=0.1755  ndcg@100=0.0516
[ckpt] epoch 160: saved metrics_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch160.json
[train] epoch 161/200  loss=0.0012  time=41.9s
[train] epoch 162/200  loss=0.0011  time=41.8s
[train] epoch 163/200  loss=0.0009  time=41.8s
[train] epoch 164/200  loss=0.0012  time=41.9s
[train] epoch 165/200  loss=0.0021  time=41.8s
[ckpt] epoch 165: saved project_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch165.npy, company_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch165.npy
[ckpt] epoch 165: saved model_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch165.pt

=== GNN [gcn] epoch 165 [p2c] K=100 ===
  royalty       recall@10=0.1888  ndcg@10=0.1054  recall@100=0.4179  ndcg@100=0.1534
  commercial    recall@10=0.1748  ndcg@10=0.0965  recall@100=0.3741  ndcg@100=0.1374
  performance   recall@10=0.1420  ndcg@10=0.0809  recall@100=0.3078  ndcg@100=0.1147

=== GNN [gcn] epoch 165 [c2p] K=100 ===
  royalty       recall@10=0.0484  ndcg@10=0.0254  recall@100=0.2161  ndcg@100=0.0588
  commercial    recall@10=0.0281  ndcg@10=0.0139  recall@100=0.1374  ndcg@100=0.0351
  performance   recall@10=0.0340  ndcg@10=0.0172  recall@100=0.1604  ndcg@100=0.0425
[ckpt] epoch 165: saved metrics_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch165.json
[train] epoch 166/200  loss=0.0014  time=41.9s
[train] epoch 167/200  loss=0.0010  time=41.9s
[train] epoch 168/200  loss=0.0009  time=41.9s
[train] epoch 169/200  loss=0.0009  time=41.8s
[train] epoch 170/200  loss=0.0008  time=41.9s
[ckpt] epoch 170: saved project_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch170.npy, company_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch170.npy
[ckpt] epoch 170: saved model_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch170.pt

=== GNN [gcn] epoch 170 [p2c] K=100 ===
  royalty       recall@10=0.3013  ndcg@10=0.1829  recall@100=0.4514  ndcg@100=0.2151
  commercial    recall@10=0.2621  ndcg@10=0.1588  recall@100=0.4167  ndcg@100=0.1920
  performance   recall@10=0.2106  ndcg@10=0.1261  recall@100=0.3468  ndcg@100=0.1553

=== GNN [gcn] epoch 170 [c2p] K=100 ===
  royalty       recall@10=0.0862  ndcg@10=0.0460  recall@100=0.2997  ndcg@100=0.0893
  commercial    recall@10=0.0746  ndcg@10=0.0386  recall@100=0.2587  ndcg@100=0.0757
  performance   recall@10=0.0520  ndcg@10=0.0261  recall@100=0.2108  ndcg@100=0.0583
[ckpt] epoch 170: saved metrics_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch170.json
[train] epoch 171/200  loss=0.0008  time=41.9s
[train] epoch 172/200  loss=0.0007  time=41.8s
[train] epoch 173/200  loss=0.0007  time=41.8s
[train] epoch 174/200  loss=0.0013  time=41.8s
[train] epoch 175/200  loss=0.0010  time=41.8s
[ckpt] epoch 175: saved project_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch175.npy, company_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch175.npy
[ckpt] epoch 175: saved model_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch175.pt

=== GNN [gcn] epoch 175 [p2c] K=100 ===
  royalty       recall@10=0.2965  ndcg@10=0.1876  recall@100=0.4519  ndcg@100=0.2220
  commercial    recall@10=0.2526  ndcg@10=0.1568  recall@100=0.4225  ndcg@100=0.1934
  performance   recall@10=0.2072  ndcg@10=0.1243  recall@100=0.3495  ndcg@100=0.1545

=== GNN [gcn] epoch 175 [c2p] K=100 ===
  royalty       recall@10=0.0741  ndcg@10=0.0380  recall@100=0.2750  ndcg@100=0.0773
  commercial    recall@10=0.0625  ndcg@10=0.0332  recall@100=0.2220  ndcg@100=0.0654
  performance   recall@10=0.0567  ndcg@10=0.0299  recall@100=0.1929  ndcg@100=0.0574
[ckpt] epoch 175: saved metrics_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch175.json
[train] epoch 176/200  loss=0.0020  time=41.9s
[train] epoch 177/200  loss=0.0012  time=41.8s
[train] epoch 178/200  loss=0.0009  time=41.8s
[train] epoch 179/200  loss=0.0007  time=41.9s
[train] epoch 180/200  loss=0.0007  time=41.9s
[ckpt] epoch 180: saved project_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch180.npy, company_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch180.npy
[ckpt] epoch 180: saved model_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch180.pt

=== GNN [gcn] epoch 180 [p2c] K=100 ===
  royalty       recall@10=0.2845  ndcg@10=0.1781  recall@100=0.4671  ndcg@100=0.2175
  commercial    recall@10=0.2461  ndcg@10=0.1462  recall@100=0.4163  ndcg@100=0.1825
  performance   recall@10=0.2232  ndcg@10=0.1345  recall@100=0.3611  ndcg@100=0.1640

=== GNN [gcn] epoch 180 [c2p] K=100 ===
  royalty       recall@10=0.1083  ndcg@10=0.0557  recall@100=0.3228  ndcg@100=0.0986
  commercial    recall@10=0.0690  ndcg@10=0.0365  recall@100=0.2595  ndcg@100=0.0748
  performance   recall@10=0.0583  ndcg@10=0.0289  recall@100=0.2241  ndcg@100=0.0631
[ckpt] epoch 180: saved metrics_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch180.json
[train] epoch 181/200  loss=0.0007  time=41.9s
[train] epoch 182/200  loss=0.0007  time=41.8s
[train] epoch 183/200  loss=0.0011  time=41.8s
[train] epoch 184/200  loss=0.0016  time=41.8s
[train] epoch 185/200  loss=0.0010  time=41.8s
[ckpt] epoch 185: saved project_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch185.npy, company_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch185.npy
[ckpt] epoch 185: saved model_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch185.pt

=== GNN [gcn] epoch 185 [p2c] K=100 ===
  royalty       recall@10=0.2767  ndcg@10=0.1754  recall@100=0.4571  ndcg@100=0.2144
  commercial    recall@10=0.2598  ndcg@10=0.1612  recall@100=0.4200  ndcg@100=0.1954
  performance   recall@10=0.2120  ndcg@10=0.1281  recall@100=0.3537  ndcg@100=0.1582

=== GNN [gcn] epoch 185 [c2p] K=100 ===
  royalty       recall@10=0.0810  ndcg@10=0.0393  recall@100=0.2708  ndcg@100=0.0769
  commercial    recall@10=0.0514  ndcg@10=0.0256  recall@100=0.2394  ndcg@100=0.0633
  performance   recall@10=0.0503  ndcg@10=0.0265  recall@100=0.2094  ndcg@100=0.0588
[ckpt] epoch 185: saved metrics_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch185.json
[train] epoch 186/200  loss=0.0008  time=41.9s
[train] epoch 187/200  loss=0.0011  time=41.8s
[train] epoch 188/200  loss=0.0008  time=41.8s
[train] epoch 189/200  loss=0.0008  time=41.8s
[train] epoch 190/200  loss=0.0009  time=41.8s
[ckpt] epoch 190: saved project_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch190.npy, company_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch190.npy
[ckpt] epoch 190: saved model_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch190.pt

=== GNN [gcn] epoch 190 [p2c] K=100 ===
  royalty       recall@10=0.2824  ndcg@10=0.1640  recall@100=0.4576  ndcg@100=0.2015
  commercial    recall@10=0.2445  ndcg@10=0.1493  recall@100=0.4150  ndcg@100=0.1857
  performance   recall@10=0.2046  ndcg@10=0.1188  recall@100=0.3547  ndcg@100=0.1505

=== GNN [gcn] epoch 190 [c2p] K=100 ===
  royalty       recall@10=0.0978  ndcg@10=0.0522  recall@100=0.2992  ndcg@100=0.0925
  commercial    recall@10=0.0626  ndcg@10=0.0317  recall@100=0.2383  ndcg@100=0.0672
  performance   recall@10=0.0547  ndcg@10=0.0285  recall@100=0.2176  ndcg@100=0.0621
[ckpt] epoch 190: saved metrics_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch190.json
[train] epoch 191/200  loss=0.0009  time=41.9s
[train] epoch 192/200  loss=0.0006  time=41.8s
[train] epoch 193/200  loss=0.0007  time=41.8s
[train] epoch 194/200  loss=0.0007  time=41.8s
[train] epoch 195/200  loss=0.0006  time=41.8s
[ckpt] epoch 195: saved project_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch195.npy, company_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch195.npy
[ckpt] epoch 195: saved model_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch195.pt

=== GNN [gcn] epoch 195 [p2c] K=100 ===
  royalty       recall@10=0.3164  ndcg@10=0.1991  recall@100=0.4639  ndcg@100=0.2311
  commercial    recall@10=0.2943  ndcg@10=0.1846  recall@100=0.4289  ndcg@100=0.2138
  performance   recall@10=0.2380  ndcg@10=0.1510  recall@100=0.3649  ndcg@100=0.1784

=== GNN [gcn] epoch 195 [c2p] K=100 ===
  royalty       recall@10=0.1073  ndcg@10=0.0584  recall@100=0.2955  ndcg@100=0.0960
  commercial    recall@10=0.0628  ndcg@10=0.0324  recall@100=0.2510  ndcg@100=0.0703
  performance   recall@10=0.0687  ndcg@10=0.0366  recall@100=0.2353  ndcg@100=0.0708
[ckpt] epoch 195: saved metrics_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch195.json
[train] epoch 196/200  loss=0.0006  time=41.9s
[train] epoch 197/200  loss=0.0006  time=41.8s
[train] epoch 198/200  loss=0.0006  time=41.8s
[train] epoch 199/200  loss=0.0006  time=41.8s
[train] epoch 200/200  loss=0.0008  time=41.8s
[ckpt] epoch 200: saved project_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch200.npy, company_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch200.npy
[ckpt] epoch 200: saved model_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch200.pt

=== GNN [gcn] epoch 200 [p2c] K=100 ===
  royalty       recall@10=0.2500  ndcg@10=0.1527  recall@100=0.4639  ndcg@100=0.1980
  commercial    recall@10=0.2414  ndcg@10=0.1455  recall@100=0.4152  ndcg@100=0.1820
  performance   recall@10=0.1851  ndcg@10=0.1114  recall@100=0.3473  ndcg@100=0.1455

=== GNN [gcn] epoch 200 [c2p] K=100 ===
  royalty       recall@10=0.1083  ndcg@10=0.0577  recall@100=0.2871  ndcg@100=0.0935
  commercial    recall@10=0.0688  ndcg@10=0.0367  recall@100=0.2603  ndcg@100=0.0752
  performance   recall@10=0.0633  ndcg@10=0.0335  recall@100=0.2115  ndcg@100=0.0637
[ckpt] epoch 200: saved metrics_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch200.json
[train_gnn] saved project z -> data/processed/project_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm.npy
[train_gnn] saved company z -> data/processed/company_z_gcn_h384_l2_hr90_c2p100_sim10_nonorm.npy
[train_gnn] saved final model -> /content/drive/MyDrive/apollo.M1.GNN/outputs/model_gcn_h384_l2_hr90_c2p100_sim10_nonorm_epoch200.pt

=== GNN [gcn] final [p2c] K=100 ===
  royalty       recall@10=0.2500  ndcg@10=0.1527  recall@100=0.4639  ndcg@100=0.1980
  commercial    recall@10=0.2414  ndcg@10=0.1455  recall@100=0.4152  ndcg@100=0.1820
  performance   recall@10=0.1851  ndcg@10=0.1114  recall@100=0.3473  ndcg@100=0.1455

=== GNN [gcn] final [c2p] K=100 ===
  royalty       recall@10=0.1083  ndcg@10=0.0577  recall@100=0.2871  ndcg@100=0.0935
  commercial    recall@10=0.0688  ndcg@10=0.0367  recall@100=0.2603  ndcg@100=0.0752
  performance   recall@10=0.0633  ndcg@10=0.0335  recall@100=0.2115  ndcg@100=0.0637

In [ ]:
  !git pull origin main && python scripts/run_sweep.py sage --hidden-dim 128 --epochs 50

From https://github.com/osung/apollo.M1.GNN
 * branch            main       -> FETCH_HEAD
Already up to date.
[sweep] model=sage  hd=128  ep=50  ckpt_every=20  out=/content/drive/MyDrive/apollo.M1.GNN/sweep/sage/sweep_sage_h128_ep50.xlsx
[sweep] 48 cells queued  (4 hard x 4 sim x 3 direction)

=== [1/48] sage_h128_hn0_sim0_p2c_only_ep50 ===
  status=OK  elapsed=432.0s
  wrote 1/48 rows -> /content/drive/MyDrive/apollo.M1.GNN/sweep/sage/sweep_sage_h128_ep50.xlsx

=== [2/48] sage_h128_hn0_sim0_c2p_only_ep50 ===
  status=OK  elapsed=386.0s
  wrote 2/48 rows -> /content/drive/MyDrive/apollo.M1.GNN/sweep/sage/sweep_sage_h128_ep50.xlsx

=== [3/48] sage_h128_hn0_sim0_both_ep50 ===
  status=OK  elapsed=398.9s
  wrote 3/48 rows -> /content/drive/MyDrive/apollo.M1.GNN/sweep/sage/sweep_sage_h128_ep50.xlsx

=== [4/48] sage_h128_hn0_sim5_p2c_only_ep50 ===
  status=OK  elapsed=544.1s
  wrote 4/48 rows -> /content/drive/MyDrive/apollo.M1.GNN/sweep/sage/sweep_sage_h128_ep50.xlsx

=== [5/48] sage_h128_

In [ ]:
  !git pull origin main && python scripts/run_sweep.py gcn --hidden-dim 128 --epochs 200

From https://github.com/osung/apollo.M1.GNN
 * branch            main       -> FETCH_HEAD
Already up to date.
[sweep] model=gcn  hd=128  ep=200  ckpt_every=20  out=/content/drive/MyDrive/apollo.M1.GNN/sweep/gcn/sweep_gcn_h128_ep200.xlsx
[sweep] 48 cells queued  (4 hard x 4 sim x 3 direction)

=== [1/48] gcn_h128_hn0_sim0_p2c_only_ep200 ===
  status=OK  elapsed=1099.6s
  wrote 1/48 rows -> /content/drive/MyDrive/apollo.M1.GNN/sweep/gcn/sweep_gcn_h128_ep200.xlsx

=== [2/48] gcn_h128_hn0_sim0_c2p_only_ep200 ===
  status=OK  elapsed=1073.4s
  wrote 2/48 rows -> /content/drive/MyDrive/apollo.M1.GNN/sweep/gcn/sweep_gcn_h128_ep200.xlsx

=== [3/48] gcn_h128_hn0_sim0_both_ep200 ===
  status=OK  elapsed=1068.0s
  wrote 3/48 rows -> /content/drive/MyDrive/apollo.M1.GNN/sweep/gcn/sweep_gcn_h128_ep200.xlsx

=== [4/48] gcn_h128_hn0_sim5_p2c_only_ep200 ===
  status=OK  elapsed=1741.4s
  wrote 4/48 rows -> /content/drive/MyDrive/apollo.M1.GNN/sweep/gcn/sweep_gcn_h128_ep200.xlsx

=== [5/48] gcn_h128_hn

In [10]:
!python scripts/run_sweep.py gcn --hidden-dim 128 --epochs 200 --resume-checkpoint "/content/drive/MyDrive/apollo.M1.GNN/sweep/gcn/model_gcn_h128_l2_hr100_c2p100_nonorm_epoch080.pt"


[sweep] resume-checkpoint parsed: (20, 0, 'both') at ep80  (file=model_gcn_h128_l2_hr100_c2p100_nonorm_epoch080.pt)
[sweep] model=gcn  hd=128  ep=200  ckpt_every=20  out=/content/drive/MyDrive/apollo.M1.GNN/sweep/gcn/sweep_gcn_h128_ep200.xlsx
[sweep] 48 cells queued  (4 hard x 4 sim x 3 direction)

=== [1/48] gcn_h128_hn0_sim0_p2c_only_ep200 ===
  reuse (metrics exist): metrics_gcn_h128_hn0_sim0_p2c_only_ep200.json

=== [2/48] gcn_h128_hn0_sim0_c2p_only_ep200 ===
  reuse (metrics exist): metrics_gcn_h128_hn0_sim0_c2p_only_ep200.json

=== [3/48] gcn_h128_hn0_sim0_both_ep200 ===
  reuse (metrics exist): metrics_gcn_h128_hn0_sim0_both_ep200.json

=== [4/48] gcn_h128_hn0_sim5_p2c_only_ep200 ===
  reuse (metrics exist): metrics_gcn_h128_hn0_sim5_p2c_only_ep200.json

=== [5/48] gcn_h128_hn0_sim5_c2p_only_ep200 ===
  reuse (metrics exist): metrics_gcn_h128_hn0_sim5_c2p_only_ep200.json

=== [6/48] gcn_h128_hn0_sim5_both_ep200 ===
  reuse (metrics exist): metrics_gcn_h128_hn0_sim5_both_ep200.js